# 🏋️ Entrenar XTTS-v2 con tu voz (afinado)

Clonar con una grabación corta (el cuaderno **Clonador de voz**) ya suena parecido. Si quieres **más parecido** en tu acento, tus muletillas y tu forma de entonar, este cuaderno **afina** XTTS-v2 con tus grabaciones. Se hace una sola vez: el modelo queda en tu Drive y lo usas en el clonador escribiendo su nombre en `modelo_afinado` (celda 3).

### Qué necesitas
- Entorno con GPU: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU** (ya viene configurado).
- **10 a 30 minutos** de tu voz (mínimo unos 3) en una carpeta de tu Drive: audios limpios, sin música ni otras voces, hablando con naturalidad, como quieres que suene el resultado. Sirven los mismos formatos que en el clonador (WAV, MP3, M4A, notas de voz, videos…).

### Pasos
1. Ejecuta la celda **1** (deja **XTTS-v2** como modelo) y las funciones **2a–2c**.
2. Celda **3**: escribe la carpeta con tus grabaciones y ejecútala. Whisper las transcribe y las corta en frases (unos minutos).
3. Celda **4**: el entrenamiento, de 10 a 40 minutos en una T4. No cierres la pestaña.
4. Celda **5**: guarda el modelo en `Mi unidad/Clonador de voz/modelos` (~1 GB).
5. En el **Clonador de voz**, escribe el nombre del modelo en `modelo_afinado` (celda 3), ejecútala y crea tu voz en la celda 4 como siempre.

Ejecuta las celdas **una por una**: cada paso tarda y conviene revisar su resultado antes del siguiente.

> **Licencia.** XTTS-v2 se distribuye con la [Coqui Public Model License](https://tts-hub.github.io/cpml): el modelo original, el que afines y los audios que generen solo admiten **uso no comercial**. La celda 3 lo asume por defecto (casilla `uso_no_comercial`). Afina únicamente con tu voz o con la de alguien que te haya dado su permiso.

In [ ]:
# @title 1 · Preparar el entorno (GPU y librerías) { display-mode: "form" }
# @markdown Elige el **modelo de voz**. **XTTS-v2** da el mejor parecido y permite entrenar con tu voz (cuaderno «Entrenar XTTS con tu voz»), pero solo para **uso no comercial**. **Chatterbox** permite **uso comercial** (licencia MIT; sus audios llevan una marca de agua inaudible).
modelo = "XTTS-v2 (uso no comercial)"  # @param ["XTTS-v2 (uso no comercial)", "Chatterbox Multilingual (uso comercial)"]
# @markdown *La primera vez tarda 1–3 minutos (Chatterbox, unos minutos más); si la vuelves a ejecutar, no reinstala nada. Para cambiar de modelo: «Entorno de ejecución → Desconectar y eliminar entorno de ejecución» y empieza otra vez.*

"""Celda 1 · Entorno: verifica la GPU, instala lo que falte (con uv, mucho más rápido que pip) y reinicia la sesión
solo si es imprescindible. También deja `instalar()` para las celdas que necesiten algo más (Whisper)."""

import importlib
import importlib.metadata as metadata
import os
import re
import shutil
import signal
import subprocess
import sys
import time

VERSION_COQUI_TTS = "0.28.0"  # versiones con las que se verificó este cuaderno
VERSION_CHATTERBOX = "0.1.7"
VERSION_FASTER_WHISPER = "1.2.1"
TORCHVISION_DE_CHATTERBOX = "0.21.0"  # la que corresponde al torch 2.6.0 que fija Chatterbox
MODELOS = {"XTTS-v2 (uso no comercial)": "xtts", "Chatterbox Multilingual (uso comercial)": "chatterbox"}
PAQUETES_EXTRA: list[str] = []  # p. ej. ["transformers<5.19"] si una actualización de Colab rompe la instalación
MODULOS_SENSIBLES = ("numpy", "scipy", "torch", "torchaudio", "transformers", "tokenizers", "huggingface_hub")


class ErrorUsuario(Exception):
    """Problema que el usuario puede corregir: se muestra como un mensaje, sin traceback."""


def confirmar_licencia(no_comercial: bool) -> None:
    """Aplica el caso de uso declarado en la casilla de XTTS-v2 y lo deja a la vista en cada ejecución (transparencia).
    La usan la celda 3 del clonador y la del cuaderno de entrenamiento."""
    if not no_comercial:
        os.environ.pop("COQUI_TOS_AGREED", None)
        raise ErrorUsuario(
            "Desmarcaste «uso_no_comercial»: la licencia de XTTS-v2 (CPML) no permite el uso comercial, así que "
            "XTTS-v2 no se usa. Si fue sin querer, vuelve a marcar la casilla y ejecuta esta celda otra vez; para "
            "uso comercial, elige Chatterbox en la celda 1 del clonador."
        )
    os.environ["COQUI_TOS_AGREED"] = "1"  # forma prevista por coqui-tts para aceptar la CPML sin preguntas interactivas
    print("ℹ️ Licencia: CPML aceptada para uso no comercial (casilla «uso_no_comercial» de esta celda).")


def registrar_errores_amigables() -> None:
    """Muestra cada ErrorUsuario en una sola línea roja en lugar de un traceback.

    IPython ignora lo que devuelve el manejador al ejecutar una celda, por eso imprime el mensaje;
    la celda igual queda marcada con error y «Ejecutar todas» se detiene ahí.
    """
    try:
        shell = get_ipython()  # noqa: F821 (la define IPython/Colab)
    except NameError:
        return
    if shell is None:
        return

    def mostrar(_shell, _tipo, valor, _traza, tb_offset=None) -> None:
        print(f"\x1b[1;31m❌ {valor}\x1b[0m", file=sys.stderr)

    shell.set_custom_exc((ErrorUsuario,), mostrar)


def version_instalada(paquete: str) -> str | None:
    """Versión instalada de un paquete, o None si no está."""
    try:
        return metadata.version(paquete)
    except metadata.PackageNotFoundError:
        return None


def version_mayor_menor(version: str) -> tuple[int, int]:
    """'2.10.0+cu126' → (2, 10)."""
    coincidencia = re.match(r"(\d+)\.(\d+)", version)
    if coincidencia is None:
        raise ValueError(f"Versión no reconocida: {version}")
    return int(coincidencia.group(1)), int(coincidencia.group(2))


def verificar_gpu() -> str:
    """Nombre y memoria de la GPU; error claro si el entorno no tiene una."""
    if shutil.which("nvidia-smi") is None:
        raise ErrorUsuario(
            "Este entorno no tiene GPU. Ve a «Entorno de ejecución → Cambiar tipo de entorno de ejecución», "
            "elige «T4 GPU», guarda y ejecuta otra vez desde la celda 1."
        )
    consulta = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
        check=False,
    )
    lineas = consulta.stdout.strip().splitlines()
    if consulta.returncode != 0 or not lineas:
        raise ErrorUsuario("La GPU no responde. Usa «Entorno de ejecución → Reiniciar sesión» y vuelve a intentarlo.")
    return lineas[0].strip()


def asegurar_ffmpeg() -> None:
    """FFmpeg convierte tu audio (MP3, M4A, OPUS, video…). Colab ya lo trae; si faltara, se instala."""
    if shutil.which("ffmpeg") is None:
        subprocess.run("apt-get -qq update && apt-get -qq install -y ffmpeg", shell=True, check=True, capture_output=True)


def especificacion_torchcodec(torch: tuple[int, int]) -> str | None:
    """torchcodec compatible con la versión de torch, según la tabla oficial de torchcodec.

    coqui-tts lo exige desde torch 2.9, porque torchaudio le delega la lectura de audio.
    """
    if torch < (2, 9):
        return None
    tabla = {(2, 9): "torchcodec==0.9.*", (2, 10): "torchcodec==0.10.*"}
    return tabla.get(torch, "torchcodec>=0.12")  # desde la 0.12 sirve para cualquier torch >= 2.11


def paquetes_pendientes(motor: str) -> list[str]:
    """Lo que falta instalar para el modelo elegido, partiendo del torch que ya trae Colab."""
    version_torch = version_instalada("torch")
    if version_torch is None:
        raise ErrorUsuario("Este entorno no tiene PyTorch. Usa un entorno estándar de Colab con GPU.")
    if motor == "chatterbox":  # trae su propio torch (2.6.0); torchvision debe ir a juego o falla al importarse
        if version_instalada("chatterbox-tts") == VERSION_CHATTERBOX:
            return list(PAQUETES_EXTRA)
        return [f"chatterbox-tts=={VERSION_CHATTERBOX}", f"torchvision=={TORCHVISION_DE_CHATTERBOX}", *PAQUETES_EXTRA]
    pendientes = []
    if version_instalada("coqui-tts") != VERSION_COQUI_TTS:
        pendientes.append(f"coqui-tts=={VERSION_COQUI_TTS}")
    if version_instalada("torchaudio") is None:
        pendientes.append(f"torchaudio=={version_torch}")  # misma versión y misma variante CUDA que torch
    codec = especificacion_torchcodec(version_mayor_menor(version_torch))
    if codec is not None and version_instalada("torchcodec") is None:
        pendientes.append(codec)
    return pendientes + list(PAQUETES_EXTRA)


def indices_pytorch(especificaciones: list[str]) -> list[str]:
    """Índices de PyTorch para las variantes CUDA pedidas (p. ej. «+cu126»), que no están publicadas en PyPI."""
    variantes = sorted({e.split("+", 1)[1] for e in especificaciones if "+" in e})
    return [f"https://download.pytorch.org/whl/{variante}" for variante in variantes]


def asegurar_uv() -> str | None:
    """Ruta de uv (instalador en Rust, varias veces más rápido que pip); si no está, se instala en segundos."""
    if shutil.which("uv") is None:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", "uv"],
            capture_output=True,
            check=False,
        )
    return shutil.which("uv")


def comandos_de_instalacion(especificaciones: list[str], indices_extra: list[str]) -> list[list[str]]:
    """Comandos a probar en orden: uv sobre el Python de esta sesión y, si falla, pip."""
    comandos = []
    uv = asegurar_uv()
    if uv is not None:
        comando = [uv, "pip", "install", "--quiet", "--python", sys.executable, "--system", "--break-system-packages"]
        for indice in indices_extra:
            comando += ["--extra-index-url", indice]
        if indices_extra:
            comando += ["--index-strategy", "unsafe-best-match"]  # combinar índices como lo hace pip
        comandos.append(comando + especificaciones)
    comando = [sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check"]
    for indice in indices_extra:
        comando += ["--extra-index-url", indice]
    comandos.append(comando + especificaciones)
    return comandos


def instalar(especificaciones: list[str], indices_extra: list[str] | None = None) -> None:
    """Instala paquetes con uv (o pip si uv falla); si nada funciona, muestra el final del registro."""
    resultado = None
    for comando in comandos_de_instalacion(especificaciones, list(indices_extra or [])):
        resultado = subprocess.run(comando, capture_output=True, text=True, check=False)
        if resultado.returncode == 0:
            importlib.invalidate_caches()  # que esta misma sesión vea los paquetes recién instalados
            return
    print(resultado.stdout[-2000:], resultado.stderr[-4000:], sep="\n")
    raise RuntimeError("No se pudo instalar: " + " ".join(especificaciones))


def instalar_si_falta(paquete: str, especificacion: str) -> None:
    """Para dependencias opcionales que solo algunas celdas usan (p. ej. faster-whisper)."""
    if version_instalada(paquete) is None:
        print(f"📦 Instalando {especificacion}…")
        instalar([especificacion])


def modulos_desactualizados() -> list[str]:
    """Librerías ya cargadas en memoria cuya versión en disco acaba de cambiar."""
    cambios = []
    for nombre in MODULOS_SENSIBLES:
        cargada = getattr(sys.modules.get(nombre), "__version__", None)
        instalada = version_instalada(nombre)
        if cargada and instalada and cargada.split("+")[0] != instalada.split("+")[0]:
            cambios.append(f"{nombre} {cargada} → {instalada}")
    return cambios


def reiniciar_sesion(cambios: list[str]) -> None:
    """Reinicia el kernel para que cargue las versiones nuevas (lo instalado se conserva)."""
    print("🔄 Se actualizaron librerías que ya estaban en memoria:", "; ".join(cambios))
    print("   Reiniciando la sesión… Cuando se reconecte, ejecuta otra vez desde la celda 1 (será inmediata).")
    sys.stdout.flush()
    os.kill(os.getpid(), signal.SIGKILL)


def preparar_entorno(eleccion: str) -> str:
    """Devuelve el motor elegido («xtts» o «chatterbox») con todo listo para usarlo."""
    inicio = time.perf_counter()
    registrar_errores_amigables()
    motor = MODELOS[eleccion]
    if motor == "xtts" and version_instalada("chatterbox-tts") is not None:
        raise ErrorUsuario(
            "Este entorno ya tiene instalado Chatterbox, que cambió las versiones de PyTorch y transformers. Para "
            "usar XTTS-v2: «Entorno de ejecución → Desconectar y eliminar entorno de ejecución» y empieza otra vez."
        )
    gpu = verificar_gpu()
    asegurar_ffmpeg()
    pendientes = paquetes_pendientes(motor)
    if pendientes:
        print("📦 Instalando:", ", ".join(pendientes), "(unos minutos)…")
        instalar(pendientes, indices_pytorch(pendientes))
    cambios = modulos_desactualizados()
    if cambios:
        reiniciar_sesion(cambios)
    paquete = "chatterbox-tts" if motor == "chatterbox" else "coqui-tts"
    print(f"✅ Entorno listo en {time.perf_counter() - inicio:.0f} s · modelo: {eleccion}")
    print(
        f"   GPU: {gpu} · Python {sys.version.split()[0]} · "
        f"torch {version_instalada('torch')} · {paquete} {version_instalada(paquete)}"
    )
    return motor


# ─── Ejecución ───
MOTOR_ELEGIDO = preparar_entorno(modelo)

In [ ]:
# @title 2a · Funciones: Drive y archivos { display-mode: "form" }
# @markdown Acceso a tu Google Drive, búsqueda de archivos por nombre y nombres de salida. No tiene parámetros: solo ejecútala.

"""Celda 2a · Archivos: Google Drive (montaje, rutas y búsqueda), lectura de .txt y archivos de salida."""

import os
import re
import shutil
import time
import unicodedata
from collections import deque
from pathlib import Path

if "ErrorUsuario" not in globals():
    raise RuntimeError("Ejecuta primero la celda 1 · Preparar el entorno.")


# ═══ Google Drive ═══════════════════════════════════════════════════════════════

RAIZ_DRIVE = Path("/content/drive")
MI_UNIDAD = RAIZ_DRIVE / "MyDrive"
CARPETA_TRABAJO = Path("/content/clonador_voz")
EXTENSIONES_MEDIOS = frozenset({
    ".wav", ".mp3", ".m4a", ".aac", ".ogg", ".oga", ".opus", ".flac", ".wma", ".amr",
    ".webm", ".mp4", ".mov", ".mkv", ".avi", ".3gp",
})
EXTENSIONES_TEXTO = frozenset({".txt", ".md"})
CARPETA_PROYECTO = "Clonador de voz"  # carpeta de Mi unidad para audios, voces guardadas y modelos entrenados
PREFIJOS_MI_UNIDAD = ("/content/drive/MyDrive/", "MyDrive/", "My Drive/", "Mi unidad/")
NOMBRES_MI_UNIDAD = frozenset({"/content/drive/MyDrive", "MyDrive", "My Drive", "Mi unidad"})


INSTRUCCIONES_DRIVE = (
    "No se pudo conectar tu Google Drive. Vuelve a ejecutar la celda y, en la ventana de Google: "
    "1) elige la MISMA cuenta con la que abriste Colab y 2) marca TODOS los permisos («Seleccionar todo») antes de "
    "«Continuar». Si sigue fallando: menú «Entorno de ejecución → Desconectar y eliminar entorno de ejecución» "
    "(Runtime → Disconnect and delete runtime) y empieza desde la celda 1, o conecta Drive con el botón de Drive del "
    "panel «Archivos» (ícono de carpeta, a la izquierda). En la celda 4 también puedes subir el audio desde tu computadora."
)


class DriveNoDisponible(ErrorUsuario):
    """Drive no se pudo montar: quien llama decide si es un error (leer un archivo) o solo un aviso (guardar)."""

    def __init__(self, detalle: str) -> None:
        super().__init__(f"{INSTRUCCIONES_DRIVE} (Detalle técnico: {detalle})")
        self.detalle = detalle


def drive_montado() -> bool:
    """¿Drive está montado de verdad? Una carpeta «MyDrive» creada a mano no cuenta."""
    return os.path.ismount(RAIZ_DRIVE) and MI_UNIDAD.is_dir()


def montar_drive(intentos: int = 2) -> Path:
    """Monta Google Drive (la primera vez pide permiso) y devuelve «Mi unidad».

    «mount failed» suele deberse a permisos incompletos, a otra cuenta o a un fallo pasajero: se reintenta una vez
    y, si persiste, se explica qué hacer.
    """
    if drive_montado():
        return MI_UNIDAD
    from google.colab import drive

    ultimo_error: Exception | None = None
    for intento in range(1, intentos + 1):
        try:
            drive.mount(str(RAIZ_DRIVE), force_remount=intento > 1)
        except Exception as error:  # noqa: BLE001 — ValueError («mount failed»), MessageError («credential propagation…»)
            ultimo_error = error
            if intento < intentos:
                print(f"⚠️ Drive no se conectó ({error}). Reintentando; si aparece la ventana de Google, marca todos los permisos.")
            continue
        if drive_montado():
            return MI_UNIDAD
        ultimo_error = RuntimeError("Drive respondió, pero «Mi unidad» no apareció")
    raise DriveNoDisponible(f"{type(ultimo_error).__name__}: {ultimo_error}")


def clave_nombre(nombre: str) -> str:
    """Forma comparable de un nombre: Unicode NFC y sin mayúsculas.

    Drive conserva tal cual los nombres de archivos subidos desde Mac, cuyas tildes vienen en forma NFD.
    """
    return unicodedata.normalize("NFC", nombre).casefold()


def ruta_en_mi_unidad(texto: str) -> Path:
    """Interpreta lo escrito como ruta dentro de «Mi unidad»; solo una ruta completa de /content sale de ella."""
    limpio = unicodedata.normalize("NFC", texto.strip().strip("\"'").strip())
    if limpio.rstrip("/") in NOMBRES_MI_UNIDAD:
        return MI_UNIDAD
    for prefijo in PREFIJOS_MI_UNIDAD:
        if limpio.startswith(prefijo):
            return MI_UNIDAD / limpio[len(prefijo):]
    if limpio.startswith("/content/"):
        return Path(limpio)  # p. ej. una unidad compartida: /content/drive/Shareddrives/…
    return MI_UNIDAD / limpio.lstrip("/")  # «/Carpeta» también es relativa a Mi unidad


def carpeta_del_proyecto(*partes: str) -> Path:
    """«Mi unidad/Clonador de voz/…» (monta Drive si hace falta y crea la carpeta)."""
    montar_drive()
    return preparar_carpeta(MI_UNIDAD.joinpath(CARPETA_PROYECTO, *partes))


def describir_ruta(ruta: Path) -> str:
    """Ruta legible: «Mi unidad/…» para Drive y la ruta completa en otro caso."""
    try:
        return f"Mi unidad/{ruta.relative_to(MI_UNIDAD)}"
    except ValueError:
        return str(ruta)


def resolver_en_drive(entrada: str, *, extensiones: frozenset[str], incluir_carpetas: bool) -> Path:
    """La ruta indicada (tolera mayúsculas, formas Unicode y extensión omitida) o, si es solo un nombre, lo busca."""
    limpio = entrada.strip().strip("\"'").strip()
    if not limpio:
        raise ErrorUsuario("Falta el nombre del archivo en Drive.")
    ruta = ruta_en_mi_unidad(limpio)
    if ruta.exists():
        return ruta
    if "/" not in limpio:
        return buscar_por_nombre(limpio, MI_UNIDAD, extensiones=extensiones, incluir_carpetas=incluir_carpetas)
    encontrada = resolver_por_partes(ruta, extensiones)
    if encontrada is None:
        raise ErrorUsuario(f"No existe «{limpio}» en Mi unidad. Revisa la ruta y el nombre del archivo.")
    print(f"📄 Usando {describir_ruta(encontrada)}")
    return encontrada


def resolver_por_partes(ruta: Path, extensiones: frozenset[str]) -> Path | None:
    """Recorre la ruta tramo a tramo: el tramo que no existe tal cual se busca entre sus hermanos por su clave."""
    actual = Path(ruta.anchor)
    partes = ruta.parts[1:]
    for indice, parte in enumerate(partes):
        exacta = actual / parte
        if exacta.exists():
            actual = exacta
            continue
        objetivo, ultimo = clave_nombre(parte), indice == len(partes) - 1
        try:
            hermanos = sorted(os.scandir(actual), key=lambda entrada: entrada.name)
        except OSError:
            return None
        coincidencia = next(
            (
                h for h in hermanos
                if clave_nombre(h.name) == objetivo or (ultimo and h.is_file() and _coincide(h.name, objetivo, extensiones))
            ),
            None,
        )
        if coincidencia is None:
            return None
        actual = Path(coincidencia.path)
    return actual


def _coincide(nombre: str, objetivo: str, extensiones: frozenset[str]) -> bool:
    """Igualdad por clave de nombre; si se omitió la extensión, acepta cualquiera de las válidas."""
    nombre = clave_nombre(nombre)
    if nombre == objetivo:
        return True
    raiz, extension = os.path.splitext(nombre)
    return raiz == objetivo and extension in extensiones


def _fecha_modificacion(ruta: Path) -> float:
    try:
        return ruta.stat().st_mtime
    except OSError:
        return 0.0


def buscar_por_nombre(
    nombre: str,
    raiz: Path,
    *,
    extensiones: frozenset[str],
    incluir_carpetas: bool,
    limite_s: float = 60.0,
    margen_s: float = 4.0,
) -> Path:
    """Búsqueda en anchura (primero lo menos profundo) con tiempo límite; ante duplicados, el más reciente."""
    objetivo = clave_nombre(nombre)
    print(f"🔎 Buscando «{nombre}» en tu Drive…")
    encontrados: list[Path] = []
    pendientes: deque[Path] = deque([raiz])
    fin = time.monotonic() + limite_s
    while pendientes and time.monotonic() < fin:
        try:
            entradas = list(os.scandir(pendientes.popleft()))
        except OSError:
            continue
        for entrada in entradas:
            if entrada.name.startswith("."):
                continue
            if entrada.is_dir(follow_symlinks=False):
                pendientes.append(Path(entrada.path))
                if incluir_carpetas and clave_nombre(entrada.name) == objetivo:
                    encontrados.append(Path(entrada.path))
            elif _coincide(entrada.name, objetivo, extensiones):
                encontrados.append(Path(entrada.path))
        if encontrados:  # margen breve para detectar homónimos y terminar
            fin = min(fin, time.monotonic() + margen_s)
    if not encontrados:
        raise ErrorUsuario(f"No encontré «{nombre}» en Mi unidad. Escribe su ruta, p. ej. «Carpeta/{nombre}».")
    elegido = max(encontrados, key=_fecha_modificacion)
    if len(encontrados) > 1:
        print(f"ℹ️ Hay {len(encontrados)} coincidencias; uso la más reciente. Para usar otra, escribe su ruta:")
        for ruta in encontrados[:5]:
            print("   •", describir_ruta(ruta))
    print(f"📄 Usando {describir_ruta(elegido)}")
    return elegido


def listar_medios(ruta: Path, maximo: int = 12, *, subcarpetas: bool = False) -> list[Path]:
    """El archivo indicado, o los audios y videos de una carpeta en orden alfabético (con sus subcarpetas, si se pide)."""
    if ruta.is_file():
        return [ruta]
    candidatos = ruta.rglob("*") if subcarpetas else ruta.iterdir()
    archivos = sorted(
        (p for p in candidatos if p.is_file() and p.suffix.casefold() in EXTENSIONES_MEDIOS and not p.name.startswith(".")),
        key=lambda p: str(p.relative_to(ruta)).casefold(),
    )
    if not archivos:
        raise ErrorUsuario(f"La carpeta «{ruta.name}» no tiene audios ni videos.")
    if len(archivos) > maximo:
        print(f"ℹ️ La carpeta tiene {len(archivos)} archivos; uso los primeros {maximo}.")
    return archivos[:maximo]


def leer_texto_de_archivo(ruta: Path) -> str:
    """UTF-8 (con o sin BOM); si no lo es, Windows-1252 (Bloc de notas antiguo) y, como último recurso, Latin-1."""
    datos = ruta.read_bytes()
    for codificacion in ("utf-8-sig", "cp1252"):
        try:
            return datos.decode(codificacion)
        except UnicodeDecodeError:
            continue
    return datos.decode("latin-1")


# ═══ Archivos de salida ═════════════════════════════════════════════════════════


def preparar_carpeta(ruta: Path, *, vaciar: bool = False) -> Path:
    """Crea la carpeta (y sus padres); con vaciar=True la deja vacía antes."""
    if vaciar and ruta.exists():
        shutil.rmtree(ruta)
    ruta.mkdir(parents=True, exist_ok=True)
    return ruta


def nombre_seguro(nombre: str, por_defecto: str = "voz_clonada") -> str:
    """Nombre de archivo válido: sin extensión de audio o subtítulos ni caracteres prohibidos."""
    base = re.sub(r"\.(wav|mp3|ogg|opus|srt)$", "", nombre.strip(), flags=re.IGNORECASE)
    base = re.sub(r'[\\/:*?"<>|\x00-\x1f]+', "_", base).strip(" ._")
    return base[:120] or por_defecto


def base_libre(carpeta: Path, base: str, extensiones: tuple[str, ...]) -> str:
    """Primer nombre (nombre, nombre_2, nombre_3…) libre con TODAS las extensiones: el audio y sus subtítulos
    quedan con el mismo nombre y nada se sobrescribe."""
    candidato, n = base, 2
    while any((carpeta / f"{candidato}{extension}").exists() for extension in extensiones):
        candidato, n = f"{base}_{n}", n + 1
    return candidato


def ruta_libre(carpeta: Path, base: str, extension: str) -> Path:
    """Primera ruta que no sobrescribe nada: nombre.ext, nombre_2.ext, nombre_3.ext…"""
    return carpeta / f"{base_libre(carpeta, base, (extension,))}{extension}"


def formatear_duracion(segundos: float) -> str:
    minutos, resto = divmod(segundos, 60.0)
    return f"{int(minutos)} min {resto:.0f} s" if minutos >= 1 else f"{resto:.1f} s"


# ─── Ejecución ───
print("✅ Funciones de archivos cargadas.")

In [ ]:
# @title 2b · Funciones: audio y volumen { display-mode: "form" }
# @markdown Conversión con FFmpeg, procesamiento del audio con NumPy, volumen parejo (LUFS) y música de fondo. No tiene parámetros: solo ejecútala.

"""Celda 2b · Audio: FFmpeg, WAV y procesamiento con NumPy, sonoridad ITU-R BS.1770-4 con limitador, remuestreo,
control de calidad (silencios), mezcla con música de fondo que se atenúa bajo la voz y reproductor."""

import subprocess
import wave
from collections.abc import Sequence
from pathlib import Path
from typing import Any

import numpy as np

if "preparar_carpeta" not in globals():
    raise RuntimeError("Ejecuta primero las celdas 1 y 2a.")


# ═══ Audio con FFmpeg ═══════════════════════════════════════════════════════════

FRECUENCIA_REFERENCIA = 22050  # Hz con los que XTTS analiza la voz de referencia


def filtros_referencia(limpiar: bool, reducir_ruido: bool) -> str:
    """Cadena de filtros de FFmpeg para preparar la voz de referencia."""
    filtros: list[str] = []
    if limpiar:
        filtros.append("highpass=f=70")  # retumbes y ruido grave
    if reducir_ruido:
        filtros.append("afftdn=nr=10:nf=-30")  # reducción espectral moderada
    if limpiar:
        recorte = "silenceremove=start_periods=1:start_duration=0.05:start_threshold=-45dB"
        filtros += [recorte, "areverse", recorte, "areverse"]  # silencio inicial y final
        filtros.append("loudnorm=I=-20:TP=-2:LRA=11")  # volumen homogéneo (EBU R128)
    return ",".join(filtros)


def ejecutar_ffmpeg(argumentos: Sequence[str], tarea: str) -> None:
    """Ejecuta FFmpeg sin ruido en pantalla; si falla, explica qué no pudo hacer."""
    comando = ["ffmpeg", "-hide_banner", "-loglevel", "error", "-nostdin", "-y", *argumentos]
    resultado = subprocess.run(comando, capture_output=True, text=True, check=False)
    if resultado.returncode != 0:
        detalle = (resultado.stderr.strip().splitlines() or ["sin detalle"])[-1]
        raise ErrorUsuario(f"FFmpeg no pudo {tarea}: {detalle}")


def preparar_referencia(
    origen: Path, destino: Path, *, filtros: str, max_segundos: float, frecuencia: int = FRECUENCIA_REFERENCIA
) -> None:
    """Cualquier audio o video → WAV mono de 16 bits a la frecuencia indicada, filtrado y recortado."""
    argumentos = ["-t", f"{max_segundos + 30:.0f}", "-i", str(origen), "-vn", "-ac", "1"]
    if filtros:
        argumentos += ["-af", filtros]
    argumentos += ["-ar", str(frecuencia), "-t", f"{max_segundos:.2f}", "-c:a", "pcm_s16le", str(destino)]
    ejecutar_ffmpeg(argumentos, f"leer «{origen.name}» como audio")


def convertir_a_mp3(origen: Path, destino: Path, kbps: int = 192) -> None:
    """WAV → MP3 a 44,1 kHz: a los 24 kHz del modelo, el MP3 no admite 192 kbps y bajaría a 160 sin avisar."""
    argumentos = ["-i", str(origen), "-ar", "44100", "-c:a", "libmp3lame", "-b:a", f"{kbps}k", str(destino)]
    ejecutar_ffmpeg(argumentos, "crear el MP3")


def convertir_a_ogg(origen: Path, destino: Path, kbps: int = 96) -> None:
    """WAV → Ogg Opus (el formato de las notas de voz): 96 kbps a 48 kHz, la frecuencia nativa de Opus."""
    argumentos = ["-i", str(origen), "-ar", "48000", "-c:a", "libopus", "-b:a", f"{kbps}k", str(destino)]
    ejecutar_ffmpeg(argumentos, "crear el OGG")


def decodificar_audio(ruta: Path, frecuencia: int, canales: int = 1, max_segundos: float | None = None) -> np.ndarray:
    """Cualquier audio o video → float32 a la frecuencia pedida, por una tubería de FFmpeg (sin archivos temporales).

    Devuelve (n,) en mono o (n, canales).
    """
    limite = ["-t", f"{max_segundos:.2f}"] if max_segundos else []
    comando = [
        "ffmpeg", "-hide_banner", "-loglevel", "error", "-nostdin", *limite, "-i", str(ruta), "-vn",
        "-ac", str(canales), "-ar", str(frecuencia), "-f", "f32le", "-",
    ]
    resultado = subprocess.run(comando, capture_output=True, check=False)
    if resultado.returncode != 0 or not resultado.stdout:
        detalle = (resultado.stderr.decode(errors="replace").strip().splitlines() or ["sin audio"])[-1]
        raise ErrorUsuario(f"FFmpeg no pudo leer «{ruta.name}» como audio: {detalle}")
    muestras = np.frombuffer(resultado.stdout, dtype="<f4").astype(np.float32)
    return muestras if canales == 1 else muestras.reshape(-1, canales)


def procesar_con_ffmpeg(x: np.ndarray, frecuencia: int, filtro: str) -> np.ndarray:
    """Aplica un filtro de FFmpeg (p. ej. «atempo=1.2») a una señal mono en memoria."""
    formato = ["-f", "f32le", "-ac", "1", "-ar", str(frecuencia)]
    comando = ["ffmpeg", "-hide_banner", "-loglevel", "error", *formato, "-i", "-", "-af", filtro, *formato, "-"]
    resultado = subprocess.run(
        comando, input=np.ascontiguousarray(x, dtype="<f4").tobytes(), capture_output=True, check=False
    )
    if resultado.returncode != 0:
        detalle = (resultado.stderr.decode(errors="replace").strip().splitlines() or ["sin detalle"])[-1]
        raise ErrorUsuario(f"FFmpeg no pudo aplicar «{filtro}»: {detalle}")
    return np.frombuffer(resultado.stdout, dtype="<f4").astype(np.float32)


# ═══ WAV y procesamiento con NumPy ══════════════════════════════════════════════


def leer_wav(ruta: Path) -> tuple[np.ndarray, int]:
    """WAV PCM de 16 bits → (muestras float32 en [-1, 1], frecuencia). Sin librerías de audio."""
    with wave.open(str(ruta), "rb") as wav:
        canales, ancho, frecuencia = wav.getnchannels(), wav.getsampwidth(), wav.getframerate()
        datos = wav.readframes(wav.getnframes())
    if ancho != 2:
        raise ValueError(f"{ruta.name}: se esperaba PCM de 16 bits")
    muestras = np.frombuffer(datos, dtype="<i2").astype(np.float32) / np.float32(32768.0)
    if canales > 1:
        muestras = muestras.reshape(-1, canales).mean(axis=1, dtype=np.float32)
    return muestras, frecuencia


def escribir_wav(ruta: Path, muestras: np.ndarray, frecuencia: int) -> None:
    """float32 en [-1, 1] → WAV PCM de 16 bits: (n,) mono o (n, canales) intercalado."""
    canales = 1 if muestras.ndim == 1 else muestras.shape[1]
    pcm = np.ascontiguousarray(np.round(np.clip(muestras, -1.0, 1.0) * 32767.0).astype("<i2"))
    with wave.open(str(ruta), "wb") as wav:
        wav.setnchannels(canales)
        wav.setsampwidth(2)
        wav.setframerate(frecuencia)
        wav.writeframes(pcm.tobytes())


def rms_por_tramas(x: np.ndarray, ventana: int, salto: int) -> np.ndarray:
    """RMS de tramas solapadas en O(n) con una suma acumulada de cuadrados (sin copiar tramas)."""
    if x.size == 0:
        return np.zeros(0)
    if x.size <= ventana:
        return np.sqrt([np.mean(np.square(x, dtype=np.float64))])
    acumulada = np.concatenate(([0.0], np.cumsum(np.square(x, dtype=np.float64))))
    inicios = np.arange(0, x.size - ventana + 1, salto)
    energia = (acumulada[inicios + ventana] - acumulada[inicios]) / ventana
    return np.sqrt(np.maximum(energia, 0.0))


def recortar_silencios(x: np.ndarray, frecuencia: int, umbral_db: float = -45.0, margen_ms: float = 60.0) -> np.ndarray:
    """Quita el silencio inicial y final: conserva de la primera a la última trama con voz."""
    ventana, salto = max(1, frecuencia // 50), max(1, frecuencia // 100)  # tramas de 20 ms cada 10 ms
    rms = rms_por_tramas(x, ventana, salto)
    if rms.size == 0 or rms.max() < 1e-5:
        return x[:0]
    activas = np.flatnonzero(rms >= rms.max() * 10.0 ** (umbral_db / 20.0))
    margen = int(frecuencia * margen_ms / 1000.0)
    inicio = max(0, int(activas[0]) * salto - margen)
    fin = min(x.size, int(activas[-1]) * salto + ventana + margen)
    return x[inicio:fin]


def aplicar_fundidos(x: np.ndarray, frecuencia: int, ms: float = 8.0) -> np.ndarray:
    """Fundido de entrada y de salida muy corto para que las uniones no hagan «clic»."""
    n = min(int(frecuencia * ms / 1000.0), x.size // 2)
    if n == 0:
        return x
    y = x.astype(np.float32, copy=True)
    rampa = np.linspace(0.0, 1.0, n, dtype=np.float32)
    y[:n] *= rampa
    y[-n:] *= rampa[::-1]
    return y


def normalizar_pico(x: np.ndarray, dbfs: float = -1.0) -> np.ndarray:
    """Escala la señal para que su pico quede en `dbfs` (−1 dBFS deja margen al MP3)."""
    pico = float(np.max(np.abs(x))) if x.size else 0.0
    if pico < 1e-6:
        return x.astype(np.float32, copy=False)
    return (x * np.float32(10.0 ** (dbfs / 20.0) / pico)).astype(np.float32, copy=False)


def ensamblar(
    fragmentos: Sequence[np.ndarray], pausas_s: Sequence[float], frecuencia: int
) -> tuple[np.ndarray, list[tuple[int, int]]]:
    """Une los fragmentos dejando tras cada uno su pausa (reserva la memoria una sola vez) y devuelve también
    dónde quedó cada uno, en muestras (inicio, fin): con eso se sincronizan los subtítulos."""
    pausas = [max(0, round(p * frecuencia)) for p in pausas_s]
    salida = np.zeros(sum(f.size for f in fragmentos) + sum(pausas), dtype=np.float32)
    posiciones: list[tuple[int, int]] = []
    posicion = 0
    for fragmento, pausa in zip(fragmentos, pausas, strict=True):
        salida[posicion : posicion + fragmento.size] = fragmento
        posiciones.append((posicion, posicion + fragmento.size))
        posicion += fragmento.size + pausa
    return salida, posiciones


def concatenar_con_pausas(fragmentos: Sequence[np.ndarray], pausas_s: Sequence[float], frecuencia: int) -> np.ndarray:
    return ensamblar(fragmentos, pausas_s, frecuencia)[0]


def remuestrear(x: np.ndarray, origen: int, destino: int) -> np.ndarray:
    """Cambia la frecuencia de muestreo de una señal mono recortando o ampliando su espectro (filtro ideal).

    El largo de la FFT es múltiplo de origen/mcd(origen, destino), así el largo de salida es exacto; el relleno
    con ceros evita que el final se «envuelva» sobre el principio.
    """
    x = np.asarray(x, dtype=np.float64)
    if origen == destino or x.size == 0:
        return x.astype(np.float32)
    divisor = np.gcd(origen, destino)
    paso = origen // divisor
    largo = -(-(x.size + origen // 50) // paso) * paso  # ≥ 20 ms de relleno
    largo_salida = largo * destino // origen
    espectro = np.fft.rfft(x, largo)
    nuevo = np.zeros(largo_salida // 2 + 1, dtype=np.complex128)
    comunes = min(espectro.size, nuevo.size)
    nuevo[:comunes] = espectro[:comunes]
    if largo_salida < largo and largo_salida % 2 == 0:
        nuevo[-1] = 0.0  # al bajar la frecuencia, el bin de Nyquist no tiene un equivalente exacto
    y = np.fft.irfft(nuevo, largo_salida) * (largo_salida / largo)
    return y[: round(x.size * destino / origen)].astype(np.float32)


def silencio_interno_maximo(x: np.ndarray, frecuencia: int, umbral_db: float = -40.0) -> float:
    """Segundos del silencio más largo ENTRE dos tramos con voz (los bordes no cuentan): delata los fragmentos en
    que el modelo se «queda callado» a mitad de frase."""
    ventana, salto = max(1, frecuencia // 50), max(1, frecuencia // 100)
    rms = rms_por_tramas(np.asarray(x, dtype=np.float32), ventana, salto)
    if rms.size == 0 or rms.max() < 1e-5:
        return 0.0
    activas = np.flatnonzero(rms >= rms.max() * 10.0 ** (umbral_db / 20.0))
    if activas.size < 2:
        return 0.0
    return float((np.diff(activas) - 1).max()) * salto / frecuencia


# ═══ Volumen: sonoridad ITU-R BS.1770-4 (EBU R128) y limitador ══════════════════

GANANCIA_MAXIMA_DB = 30.0  # no se amplifica más que esto aunque el audio venga muy bajo


def coeficientes_filtro_k(frecuencia: int) -> tuple[tuple[np.ndarray, np.ndarray], tuple[np.ndarray, np.ndarray]]:
    """(b, a) de las dos etapas del filtro K (estante de agudos + pasa altos) para cualquier frecuencia de muestreo.

    Fórmulas de libebur128; a 48 kHz reproducen los coeficientes tabulados en la norma.
    """
    f0, ganancia_db, q = 1681.974450955533, 3.999843853973347, 0.7071752369554196
    k = np.tan(np.pi * f0 / frecuencia)
    vh = 10.0 ** (ganancia_db / 20.0)
    vb = vh**0.4996667741545416
    a0 = 1.0 + k / q + k * k
    estante = (
        np.array([(vh + vb * k / q + k * k) / a0, 2.0 * (k * k - vh) / a0, (vh - vb * k / q + k * k) / a0]),
        np.array([1.0, 2.0 * (k * k - 1.0) / a0, (1.0 - k / q + k * k) / a0]),
    )
    f0, q = 38.13547087602444, 0.5003270373238773
    k = np.tan(np.pi * f0 / frecuencia)
    a0 = 1.0 + k / q + k * k
    pasa_altos = (np.array([1.0, -2.0, 1.0]), np.array([1.0, 2.0 * (k * k - 1.0) / a0, (1.0 - k / q + k * k) / a0]))
    return estante, pasa_altos


def largo_fft_rapido(n: int) -> int:
    """Menor 2^a·3^b·5^c ≥ n: con esos tamaños la FFT es rápida y casi no se rellena de más."""
    mejor = 1 << max(0, n - 1).bit_length()
    potencia5 = 1
    while potencia5 < mejor:
        potencia35 = potencia5
        while potencia35 < mejor:
            veces = -(-n // potencia35)  # cuántas veces cabe: se completa con la menor potencia de 2 suficiente
            mejor = min(mejor, potencia35 << (veces - 1).bit_length())
            potencia35 *= 3
        potencia5 *= 5
    return mejor


def ponderar_k(x: np.ndarray, frecuencia: int) -> np.ndarray:
    """Aplica el filtro K multiplicando espectros: con 0,5 s de relleno equivale al filtro IIR, sin bucles en Python.
    Acepta (n,) o (n, canales): cada canal se filtra por separado."""
    largo = largo_fft_rapido(x.shape[0] + frecuencia // 2)
    z = np.exp(-2j * np.pi * np.arange(largo // 2 + 1) / largo)  # z⁻¹ en cada frecuencia de la FFT
    respuesta = np.ones_like(z)
    for b, a in coeficientes_filtro_k(frecuencia):
        respuesta *= (b[0] + z * (b[1] + z * b[2])) / (a[0] + z * (a[1] + z * a[2]))
    if x.ndim == 2:
        respuesta = respuesta[:, None]
    return np.fft.irfft(np.fft.rfft(x, largo, axis=0) * respuesta, largo, axis=0)[: x.shape[0]]


def sonoridad_integrada(x: np.ndarray, frecuencia: int) -> float:
    """Sonoridad integrada (LUFS) de una señal mono o estéreo: bloques de 400 ms con 75 % de solapamiento,
    compuerta absoluta de −70 LUFS y relativa de −10 LU; los canales suman su energía (peso 1, como L y R en la
    norma). Devuelve −inf si todo es silencio."""
    y = ponderar_k(np.asarray(x, dtype=np.float64), frecuencia)
    cuadrados = y * y if y.ndim == 1 else (y * y).sum(axis=1)
    ventana, salto = round(0.4 * frecuencia), round(0.1 * frecuencia)
    acumulada = np.concatenate(([0.0], np.cumsum(cuadrados)))
    if cuadrados.size >= ventana:
        inicios = np.arange(0, cuadrados.size - ventana + 1, salto)
        energias = (acumulada[inicios + ventana] - acumulada[inicios]) / ventana
    else:
        energias = np.array([acumulada[-1] / max(1, cuadrados.size)])
    energias = np.maximum(energias, 1e-20)
    sonoridades = -0.691 + 10.0 * np.log10(energias)
    energias, sonoridades = energias[sonoridades > -70.0], sonoridades[sonoridades > -70.0]
    if energias.size == 0:
        return float("-inf")
    compuerta = -0.691 + 10.0 * np.log10(energias.mean()) - 10.0
    return float(-0.691 + 10.0 * np.log10(energias[sonoridades > compuerta].mean()))


def minimo_deslizante(x: np.ndarray, ventana: int) -> np.ndarray:
    """m[i] = min(x[i : i + ventana]) en O(n) con el algoritmo de van Herk/Gil-Werman, todo vectorizado:
    cada ventana cae en dos bloques consecutivos, y basta el mínimo-sufijo de uno y el mínimo-prefijo del otro."""
    if ventana <= 1 or x.size == 0:
        return x.astype(np.float64, copy=True)
    bloques = -(-(x.size + ventana - 1) // ventana)
    relleno = np.full(bloques * ventana, np.inf)
    relleno[: x.size] = x
    matriz = relleno.reshape(bloques, ventana)
    prefijo = np.minimum.accumulate(matriz, axis=1).ravel()
    sufijo = np.minimum.accumulate(matriz[:, ::-1], axis=1)[:, ::-1].ravel()
    indices = np.arange(x.size)
    return np.minimum(sufijo[indices], prefijo[indices + ventana - 1])


def limitar_picos(x: np.ndarray, frecuencia: int, techo_dbfs: float = -1.0) -> np.ndarray:
    """Limitador con anticipación: la ganancia baja en rampa antes de cada pico y ninguna muestra pasa el techo.

    g[n] = media de los últimos 7,5 ms de min(ganancia necesaria en los próximos 15 ms): como la media abarca
    menos que el mínimo, en cada pico la ganancia nunca supera la necesaria. En estéreo, una sola ganancia para
    ambos canales (manda el más alto), para que la imagen no se mueva.
    """
    techo = 10.0 ** (techo_dbfs / 20.0)
    x = np.asarray(x)
    amplitud = np.abs(np.asarray(x, dtype=np.float64))
    if amplitud.ndim == 2:
        amplitud = amplitud.max(axis=1)
    if amplitud.size == 0 or amplitud.max() <= techo:
        return np.asarray(x, dtype=np.float32)
    anticipacion, suavizado = max(1, round(0.015 * frecuencia)), max(1, round(0.0075 * frecuencia))
    envolvente = minimo_deslizante(np.minimum(1.0, techo / np.maximum(amplitud, 1e-12)), anticipacion)
    acumulada = np.concatenate(([0.0], np.cumsum(envolvente)))
    fin = np.arange(1, amplitud.size + 1)
    inicio = np.maximum(0, fin - suavizado)
    ganancia = (acumulada[fin] - acumulada[inicio]) / (fin - inicio)
    if x.ndim == 2:
        ganancia = ganancia[:, None]
    return np.clip(x * ganancia, -techo, techo).astype(np.float32)  # el recorte solo absorbe redondeos


def nivelar_volumen(x: np.ndarray, frecuencia: int, objetivo_lufs: float, techo_dbfs: float = -1.0) -> np.ndarray:
    """Lleva la sonoridad integrada al objetivo (volumen percibido parejo) y limita los picos al techo."""
    actual = sonoridad_integrada(x, frecuencia)
    if not np.isfinite(actual):
        return normalizar_pico(x, techo_dbfs)
    ganancia_db = min(objetivo_lufs - actual, GANANCIA_MAXIMA_DB)
    return limitar_picos(np.asarray(x, dtype=np.float64) * 10.0 ** (ganancia_db / 20.0), frecuencia, techo_dbfs)


# ═══ Música de fondo con atenuación automática bajo la voz («ducking») ══════════

ATENUACION_MUSICA_DB = 12.0  # cuánto baja la música mientras hay voz
ENTRADA_MUSICA_S = 1.5  # la música suena sola antes de la primera palabra…
SALIDA_MUSICA_S = 2.5  # …y se desvanece después de la última
TRAMA_S = 0.01


def actividad_de_voz(x: np.ndarray, frecuencia: int, umbral_db: float = -35.0) -> np.ndarray:
    """¿Hay voz en cada trama de 10 ms? (RMS de 20 ms frente al de la trama más fuerte)."""
    ventana, salto = max(1, round(2 * TRAMA_S * frecuencia)), max(1, round(TRAMA_S * frecuencia))
    rms = rms_por_tramas(np.asarray(x, dtype=np.float32), ventana, salto)
    if rms.size == 0 or rms.max() < 1e-5:
        return np.zeros(rms.size, dtype=bool)
    return rms >= rms.max() * 10.0 ** (umbral_db / 20.0)


def dilatar(activas: np.ndarray, antes: int, despues: int) -> np.ndarray:
    """Extiende cada tramo activo `antes` tramas hacia atrás y `despues` hacia adelante (sumas acumuladas, O(n))."""
    acumulada = np.concatenate(([0], np.cumsum(activas, dtype=np.int64)))
    indices = np.arange(activas.size)
    desde = np.clip(indices - despues, 0, activas.size)
    hasta = np.clip(indices + antes + 1, 0, activas.size)
    return acumulada[hasta] - acumulada[desde] > 0


def ganancia_de_atenuacion(activas: np.ndarray, atenuacion_db: float, ataque_s: float = 0.08, relajacion_s: float = 0.6) -> np.ndarray:
    """Ganancia en dB por trama: baja rápido cuando empieza la voz y sube despacio cuando termina (sin «bombeo»)."""
    objetivo = np.where(dilatar(activas, antes=round(0.15 / TRAMA_S), despues=round(0.3 / TRAMA_S)), -atenuacion_db, 0.0)
    ataque, relajacion = np.exp(-TRAMA_S / ataque_s), np.exp(-TRAMA_S / relajacion_s)
    ganancia = np.empty_like(objetivo)
    actual = 0.0
    for i, meta in enumerate(objetivo.tolist()):  # filtro de un polo asimétrico: no se puede vectorizar
        actual = meta + (ataque if meta < actual else relajacion) * (actual - meta)
        ganancia[i] = actual
    return ganancia


def mezclar_con_musica(
    voz: np.ndarray,
    musica: np.ndarray,
    frecuencia: int,
    *,
    nivel_db: float,
    atenuacion_db: float = ATENUACION_MUSICA_DB,
    entrada_s: float = ENTRADA_MUSICA_S,
    salida_s: float = SALIDA_MUSICA_S,
) -> np.ndarray:
    """Voz (n,) + música (m,) o (m, canales) → mezcla con la música en bucle, `nivel_db` LU por debajo de la voz
    en las pausas y `atenuacion_db` más abajo mientras se habla; con entrada y salida de música y fundidos.

    Devuelve (entrada + n + salida, canales); la voz va al centro.
    """
    musica = np.asarray(musica, dtype=np.float32)
    canales = 1 if musica.ndim == 1 else musica.shape[1]
    musica = musica.reshape(musica.shape[0], canales)
    if musica.shape[0] == 0:
        raise ErrorUsuario("La música de fondo no tiene audio.")
    entrada, salida = round(entrada_s * frecuencia), round(salida_s * frecuencia)
    total = entrada + voz.size + salida
    vueltas = -(-total // musica.shape[0])
    cama = np.tile(musica, (vueltas, 1))[:total].astype(np.float64)
    pista = np.zeros(total, dtype=np.float64)
    pista[entrada : entrada + voz.size] = voz
    s_voz, s_musica = sonoridad_integrada(voz, frecuencia), sonoridad_integrada(cama, frecuencia)
    if np.isfinite(s_voz) and np.isfinite(s_musica):
        cama *= 10.0 ** ((s_voz + nivel_db - s_musica) / 20.0)
    ganancia_db = ganancia_de_atenuacion(actividad_de_voz(pista, frecuencia), atenuacion_db)
    centros = (np.arange(ganancia_db.size) + 1.0) * TRAMA_S  # centro de cada ventana de 20 ms
    ganancia = 10.0 ** (np.interp(np.arange(total) / frecuencia, centros, ganancia_db) / 20.0)
    fundido = np.ones(total)
    rampa = min(entrada, total // 2) or min(frecuencia, total // 2)
    fundido[:rampa] = np.linspace(0.0, 1.0, rampa)
    caida = min(salida, total // 2)
    if caida:
        fundido[total - caida :] = np.linspace(1.0, 0.0, caida)
    mezcla = cama * (ganancia * fundido)[:, None] + pista[:, None]
    return mezcla.astype(np.float32)


# ═══ Whisper: transcribir lo que se oye ═════════════════════════════════════════

MODELO_WHISPER = "large-v3-turbo"  # rápido, para verificar fragmentos (celda 6)


def obtener_whisper(nombre: str = MODELO_WHISPER) -> Any:
    """faster-whisper en la GPU: se instala y se carga solo la primera vez que hace falta."""
    modelos = globals().setdefault("_WHISPER", {})
    if nombre not in modelos:
        instalar_si_falta("faster-whisper", f"faster-whisper=={VERSION_FASTER_WHISPER}")
        from faster_whisper import WhisperModel

        print(f"⏳ Cargando Whisper ({nombre}) en la GPU (la primera vez lo descarga)…")
        modelos[nombre] = WhisperModel(nombre, device="cuda", compute_type="float16")
    return modelos[nombre]


def liberar_whisper() -> None:
    """Libera la memoria de GPU de los modelos de Whisper cargados."""
    globals().get("_WHISPER", {}).clear()
    import gc

    gc.collect()
    try:
        import torch
    except ImportError:
        return
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def transcribir(onda: np.ndarray, frecuencia: int, idioma: str) -> str:
    """Lo que Whisper entiende de un fragmento (decodificación voraz: rápida y determinista)."""
    audio = remuestrear(onda, frecuencia, 16000)
    trozos, _ = obtener_whisper().transcribe(
        audio, language=idioma.split("-")[0], beam_size=1, condition_on_previous_text=False, without_timestamps=True
    )
    return " ".join(trozo.text.strip() for trozo in trozos)


LIMITE_REPRODUCTOR_MB = 8.0  # el reproductor incrusta el audio en el cuaderno: más grande lo vuelve pesado
SEGUNDOS_VISTA_PREVIA = 60


def mostrar_reproductor(ruta: Path) -> None:
    """Reproductor dentro de la celda; si el archivo es grande, solo suena su primer minuto."""
    from IPython.display import Audio, display

    if ruta.stat().st_size > LIMITE_REPRODUCTOR_MB * 1_000_000:
        vista = preparar_carpeta(CARPETA_TRABAJO / "vistas_previas") / f"{ruta.stem}.mp3"
        argumentos = ["-t", str(SEGUNDOS_VISTA_PREVIA), "-i", str(ruta), "-c:a", "libmp3lame", "-b:a", "96k", str(vista)]
        ejecutar_ffmpeg(argumentos, "crear la vista previa")
        print("ℹ️ Audio largo: aquí suena el primer minuto; el archivo completo está guardado.")
        ruta = vista
    display(Audio(filename=str(ruta)))


# ─── Ejecución ───
print("✅ Funciones de audio cargadas.")

In [ ]:
# @title 2c · Funciones: texto, voces, subtítulos y pronunciación { display-mode: "form" }
# @markdown División en frases y pausas, idiomas y voces mezclados, lotes, subtítulos, montos y cifras en español, reglas de pronunciación y comparación de textos. No tiene parámetros: solo ejecútala.

"""Celda 2c · Texto: frases y pausas, tramos por idioma y por voz, bloques (lotes), subtítulos .srt, montos y cifras
en español, diccionario de pronunciación y tasa de error por caracteres (para verificar lo que se oye)."""

import re
import unicodedata
from collections.abc import Sequence
from dataclasses import dataclass


# ═══ Texto ══════════════════════════════════════════════════════════════════════


@dataclass(frozen=True)
class Segmento:
    """Trozo de texto que el modelo lee de una vez, la pausa (en segundos) que va después, su idioma y su voz.

    `visible` es ese trozo tal como se escribió (para los subtítulos) y `grupo` une los trozos que salieron de una
    misma frase escrita cuando las cifras o las reglas de pronunciación la alargaron más allá del límite del modelo.
    """

    texto: str
    pausa: float
    idioma: str = ""  # vacío: el idioma principal elegido en la celda 5
    voz: str = ""  # vacío: la voz principal (celda 4)
    visible: str = ""
    grupo: int = 0


PATRON_PAUSA = re.compile(r"\[\s*pausa(?:\s*[=:]?\s*(\d+(?:[.,]\d+)?)\s*s?)?\s*\]", re.IGNORECASE)
PATRON_FIN_FRASE = re.compile(r"(?<=[.!?…।؟])\s+")
PATRON_CORTE_SUAVE = re.compile(r"(?<=[,;:—–])\s+")
PAUSA_MARCA = 1.0  # segundos de un [pausa] sin número
MINIMO_CARACTERES = 20  # frases más cortas se unen a la siguiente: XTTS falla con textos muy breves
# Un punto tras estas palabras (o tras una sola letra: «p. ej.», «J. K. Rowling») no cierra la frase.
ABREVIATURAS = frozenset({
    "sr", "sra", "srta", "sres", "dr", "dra", "dres", "ing", "lic", "mg", "mag", "mtro", "mtra", "prof", "profa",
    "arq", "abg", "abog", "gral", "cnel", "sto", "sta", "mons", "pdte", "ud", "uds", "mr", "mrs", "ms", "st", "jr",
    "av", "avda", "ca", "cdra", "mz", "lt", "urb", "dpto", "depto", "psje", "nro", "núm", "num", "pág", "págs", "pag",
    "fig", "vol", "ej", "aprox", "máx", "mín", "tel", "cel", "cía", "inc", "vs", "e.g", "i.e", "s/",
})


def pausa_de_parrafo(pausa_frase: float) -> float:
    return max(0.6, 2.0 * pausa_frase)


def dividir_texto(texto: str, limite: int, pausa_frase: float) -> list[Segmento]:
    """Texto → segmentos que el modelo puede leer, con la pausa que va tras cada uno.

    [pausa] o [pausa 2.5] fija un silencio; cada salto de línea es un párrafo (pausa más larga).
    """
    pausa_parrafo = pausa_de_parrafo(pausa_frase)
    segmentos: list[Segmento] = []
    for bloque, pausa_marca in _bloques_por_marcas(texto):
        parrafos = [p for p in re.split(r"\n+", bloque) if p.strip()]
        for i, parrafo in enumerate(parrafos):
            frases = _frases(parrafo, limite)
            for j, frase in enumerate(frases):
                if j < len(frases) - 1:
                    pausa = pausa_frase
                else:
                    pausa = pausa_parrafo if i < len(parrafos) - 1 else 0.0
                segmentos.append(Segmento(frase, pausa))
        if segmentos and pausa_marca:
            segmentos[-1] = Segmento(segmentos[-1].texto, segmentos[-1].pausa + pausa_marca)
    return segmentos


def _bloques_por_marcas(texto: str) -> list[tuple[str, float]]:
    """Corta el texto en cada marca de pausa; devuelve (bloque, segundos de la marca que lo sigue)."""
    bloques, inicio = [], 0
    for marca in PATRON_PAUSA.finditer(texto):
        segundos = float(marca.group(1).replace(",", ".")) if marca.group(1) else PAUSA_MARCA
        bloques.append((texto[inicio : marca.start()], segundos))
        inicio = marca.end()
    bloques.append((texto[inicio:], 0.0))
    return bloques


def pausa_inicial(texto: str) -> float:
    """Segundos de las marcas [pausa] que aparecen antes de la primera palabra (todas, si no hay palabras)."""
    total = 0.0
    for bloque, segundos in _bloques_por_marcas(texto):
        if any(caracter.isalnum() for caracter in bloque):
            break
        total += segundos
    return total


def partir_lectura(texto: str, limite: int) -> list[str]:
    """Trozos que el modelo puede leer (sin pausas ni párrafos): sirve para volver a partir una frase que las cifras
    o la pronunciación alargaron."""
    return _frases(texto, limite)


def _frases(parrafo: str, limite: int) -> list[str]:
    """Frases de un párrafo: corta en . ! ? …, parte las demasiado largas y une las muy cortas."""
    limpio = re.sub(r"\s+", " ", parrafo).strip()
    piezas: list[str] = []
    for frase in _dividir_en_frases(limpio):
        piezas.extend(_partir_frase_larga(frase, limite))
    return [frase for frase in _unir_cortas(piezas, limite) if any(c.isalnum() for c in frase)]  # nada de «.» sueltos


def _dividir_en_frases(texto: str) -> list[str]:
    """Corta tras . ! ? … salvo cuando el punto cierra una abreviatura («Dr. Pérez», «Av. Grau», «S/. 150»)."""
    frases: list[str] = []
    inicio = 0
    for corte in PATRON_FIN_FRASE.finditer(texto):
        if _termina_en_abreviatura(texto[inicio : corte.start()]):
            continue
        frases.append(texto[inicio : corte.start()])
        inicio = corte.end()
    frases.append(texto[inicio:])
    return [frase for frase in frases if frase]


def _termina_en_abreviatura(texto: str) -> bool:
    """¿El punto final de `texto` cierra una abreviatura y no una frase?"""
    if not texto.endswith("."):
        return False
    palabra = texto.rsplit(maxsplit=1)[-1].lstrip("([«“\"'¿¡").rstrip(".").casefold()
    return (len(palabra) == 1 and palabra.isalpha()) or palabra in ABREVIATURAS


def _partir_frase_larga(frase: str, limite: int) -> list[str]:
    """Una frase que excede el límite se parte primero en , ; : — y, si no basta, entre palabras."""
    if len(frase) <= limite:
        return [frase]
    trozos: list[str] = []
    for pieza in PATRON_CORTE_SUAVE.split(frase):
        trozos.extend(_partir_por_palabras(pieza, limite) if len(pieza) > limite else [pieza])
    return _empaquetar(trozos, limite)


def _partir_por_palabras(texto: str, limite: int) -> list[str]:
    palabras: list[str] = []
    for palabra in texto.split():
        while len(palabra) > limite:  # una «palabra» enorme (URL, etc.) se corta en seco
            palabras.append(palabra[:limite])
            palabra = palabra[limite:]
        palabras.append(palabra)
    return _empaquetar(palabras, limite)


def _empaquetar(piezas: Sequence[str], limite: int) -> list[str]:
    """Une piezas consecutivas con espacios sin pasar del límite (voraz, respeta el orden)."""
    grupos: list[str] = []
    actual = ""
    for pieza in piezas:
        candidato = f"{actual} {pieza}" if actual else pieza
        if len(candidato) <= limite:
            actual = candidato
        else:
            if actual:
                grupos.append(actual)
            actual = pieza
    if actual:
        grupos.append(actual)
    return grupos


def _unir_cortas(frases: Sequence[str], limite: int) -> list[str]:
    """Une las frases de menos de MINIMO_CARACTERES con la siguiente (o con la anterior si es la última)."""
    resultado: list[str] = []
    pendiente = ""
    for frase in frases:
        if pendiente and len(pendiente) + 1 + len(frase) > limite:
            resultado.append(pendiente)
            pendiente = ""
        actual = f"{pendiente} {frase}" if pendiente else frase
        if len(actual) < MINIMO_CARACTERES:
            pendiente = actual
        else:
            resultado.append(actual)
            pendiente = ""
    if pendiente:
        if resultado and len(resultado[-1]) + 1 + len(pendiente) <= limite:
            resultado[-1] = f"{resultado[-1]} {pendiente}"
        else:
            resultado.append(pendiente)
    return resultado


def partir_por_la_mitad(texto: str) -> tuple[str, str] | None:
    """Corta en el espacio más cercano al centro; sirve para reintentar un fragmento que el modelo rechazó."""
    espacios = [i for i, caracter in enumerate(texto) if caracter == " "]
    if not espacios:
        return None
    corte = min(espacios, key=lambda i: abs(i - len(texto) // 2))
    return texto[:corte].strip(), texto[corte + 1 :].strip()


# ═══ Etiquetas: idiomas y voces ════════════════════════════════════════════════

# Idiomas de los formularios (celda 5 del clonador y celda 3 del entrenamiento). XTTS-v2 también habla chino,
# japonés y coreano, pero requieren pip install "coqui-tts[zh,ja,ko]"; Chatterbox no habla checo ni húngaro.
IDIOMAS = {
    "Español": "es", "English": "en", "Português": "pt", "Français": "fr", "Italiano": "it",
    "Deutsch": "de", "Nederlands": "nl", "Polski": "pl", "Türkçe": "tr", "Русский": "ru",
    "Čeština": "cs", "Magyar": "hu", "العربية": "ar", "हिन्दी": "hi",
}

# [en] … [/en] cambia el idioma; [voz: Nombre] … [/voz] cambia la voz (diálogos). Las de pausa no coinciden.
PATRON_ETIQUETA = re.compile(
    r"\[\s*(?:(?P<cierre>/)?(?P<idioma>[a-z]{2}(?:-[a-z]{2})?)|/\s*voz|voz(?:\s*[:=]\s*(?P<voz>[^\]\n]*))?)\s*\]",
    re.IGNORECASE,
)
PUNTUACION_INICIAL = re.compile(r"^\s*[.,;:!?…»”\"')\]]+")


@dataclass(frozen=True)
class Tramo:
    """Parte continua del texto que se lee en un mismo idioma y con una misma voz."""

    texto: str
    idioma: str
    voz: str = ""  # vacío: la voz principal


def dividir_en_tramos(texto: str, principal: str, disponibles: frozenset[str]) -> tuple[list[Tramo], list[str]]:
    """«Hola [en]Hello![/en] [voz: Ana]¿Qué tal?[/voz]» → tramos (es), (en), (es, Ana).

    `[xx]` pasa al idioma xx y `[/xx]` vuelve al principal; `[voz: Nombre]` cambia de voz y `[/voz]` vuelve a la
    principal. Idioma y voz son independientes; una etiqueta sin cierre vale hasta la siguiente o hasta el final.
    Un código de idioma no disponible se avisa y se lee en el idioma principal.
    """
    tramos: list[Tramo] = []
    avisos: list[str] = []
    idioma, voz, inicio = principal, "", 0
    for etiqueta in PATRON_ETIQUETA.finditer(texto):
        _agregar_tramo(tramos, texto[inicio : etiqueta.start()], idioma, voz)
        inicio = etiqueta.end()
        if etiqueta.group("idioma") is None:
            voz = " ".join((etiqueta.group("voz") or "").split())
            continue
        codigo = etiqueta.group("idioma").lower()
        if etiqueta.group("cierre") or codigo in disponibles:
            idioma = principal if etiqueta.group("cierre") else codigo
        else:
            avisos.append(f"Idioma «{codigo}» no disponible ({etiqueta.group(0)}): esa parte se lee en el idioma principal.")
            idioma = principal
    _agregar_tramo(tramos, texto[inicio:], idioma, voz)
    return tramos, avisos


def _agregar_tramo(tramos: list[Tramo], texto: str, idioma: str, voz: str) -> None:
    """Agrega un tramo. La puntuación que sigue a un cierre («[/en].») y los espacios sueltos van al tramo anterior,
    para no perder la entonación final ni los saltos de párrafo."""
    if tramos:
        anterior = tramos[-1]
        puntuacion = PUNTUACION_INICIAL.match(texto)
        if puntuacion:
            anterior = Tramo(anterior.texto + puntuacion.group(0).strip(), anterior.idioma, anterior.voz)
            texto = texto[puntuacion.end() :]
        if not texto.strip() or (anterior.idioma, anterior.voz) == (idioma, voz):
            tramos[-1] = Tramo(anterior.texto + texto, anterior.idioma, anterior.voz)
            return
        tramos[-1] = anterior
    if texto.strip():
        tramos.append(Tramo(texto, idioma, voz))


def hay_salto_de_parrafo(anterior: str, siguiente: str) -> bool:
    """¿Hay un salto de línea entre el final de un tramo y el inicio del siguiente?"""
    return "\n" in anterior[len(anterior.rstrip()) :] + siguiente[: len(siguiente) - len(siguiente.lstrip())]


# ═══ Lotes: un audio por bloque ═════════════════════════════════════════════════

PATRON_SEPARADOR = re.compile(r"^[ \t]*-{3,}[ \t]*$", re.MULTILINE)  # una línea con solo «---»
PATRON_TITULO = re.compile(r"\A\s*#[ \t]*(?P<titulo>[^\n]*)(?:\n|\Z)")


@dataclass(frozen=True)
class Bloque:
    """Parte del texto que se convierte en un audio aparte; `titulo` (opcional) da nombre al archivo."""

    titulo: str
    texto: str


def dividir_en_bloques(texto: str) -> list[Bloque]:
    """Bloques separados por líneas «---». Si la primera línea de un bloque es «# Título», nombra el archivo y no
    se lee. Se descartan los bloques sin nada pronunciable."""
    bloques: list[Bloque] = []
    for crudo in PATRON_SEPARADOR.split(texto):
        titulo = ""
        encabezado = PATRON_TITULO.match(crudo)
        if encabezado:
            titulo, crudo = encabezado.group("titulo").strip(), crudo[encabezado.end() :]
        if any(caracter.isalnum() for caracter in crudo):
            bloques.append(Bloque(titulo, crudo))
    return bloques


# ═══ Subtítulos .srt ════════════════════════════════════════════════════════════

ANCHO_SUBTITULO = 42  # caracteres por línea (pauta habitual de subtitulado)
LINEAS_SUBTITULO = 2


def _corte_feo(palabra: str) -> bool:
    """Una línea no debe terminar en una palabra de 1–2 caracteres («a», «el», «S/»): queda colgando."""
    return len(palabra) <= 2


def _lineas_voraces(palabras: Sequence[str], ancho: int) -> list[list[str]]:
    """Reparte palabras en líneas de hasta `ancho` caracteres (una palabra más larga ocupa su propia línea); si una
    línea terminaría en palabras cortas («de la»), pasan a la siguiente."""
    lineas: list[list[str]] = []
    largo = 0
    for palabra in palabras:
        if lineas and largo + 1 + len(palabra) <= ancho:
            lineas[-1].append(palabra)
            largo += 1 + len(palabra)
            continue
        arrastre: list[str] = []
        while (
            lineas
            and len(lineas[-1]) > 1
            and _corte_feo(lineas[-1][-1])
            and len(" ".join([lineas[-1][-1], *arrastre, palabra])) <= ancho
        ):
            arrastre.insert(0, lineas[-1].pop())
        lineas.append(arrastre + [palabra])
        largo = len(" ".join(lineas[-1]))
    return lineas


def _en_dos_lineas(palabras: Sequence[str], ancho: int) -> str:
    """Texto de un subtítulo: una línea si cabe; si no, dos lo más parejas posible y sin dejar colgando una palabra
    corta al final de la primera."""
    texto = " ".join(palabras)
    if len(texto) <= ancho or len(palabras) < 2:
        return texto
    largos = [len(" ".join(palabras[:corte])) for corte in range(1, len(palabras))]

    def costo(corte: int) -> int:
        return max(largos[corte - 1], len(texto) - largos[corte - 1] - 1) + (ancho if _corte_feo(palabras[corte - 1]) else 0)

    corte = min(range(1, len(palabras)), key=costo)
    return f"{' '.join(palabras[:corte])}\n{' '.join(palabras[corte:])}"


def _tiempo_srt(segundos: float) -> str:
    milesimas = max(0, round(segundos * 1000))
    horas, milesimas = divmod(milesimas, 3_600_000)
    minutos, milesimas = divmod(milesimas, 60_000)
    segundos_enteros, milesimas = divmod(milesimas, 1000)
    return f"{horas:02d}:{minutos:02d}:{segundos_enteros:02d},{milesimas:03d}"


def subtitulos_srt(entradas: Sequence[tuple[float, float, str]]) -> str:
    """(inicio s, fin s, texto) → archivo .srt. Un texto que no cabe en 2 líneas de 42 caracteres se reparte en
    varios subtítulos, con el tiempo dividido en proporción a sus caracteres."""
    subtitulos: list[tuple[float, float, str]] = []
    for inicio, fin, texto in entradas:
        lineas = _lineas_voraces(texto.split(), ANCHO_SUBTITULO)
        grupos = [sum(lineas[i : i + LINEAS_SUBTITULO], []) for i in range(0, len(lineas), LINEAS_SUBTITULO)]
        total = sum(len(" ".join(grupo)) for grupo in grupos)
        momento = inicio
        for grupo in grupos:
            final = momento + (fin - inicio) * len(" ".join(grupo)) / total
            subtitulos.append((momento, final, _en_dos_lineas(grupo, ANCHO_SUBTITULO)))
            momento = final
    bloques = [
        f"{numero}\n{_tiempo_srt(inicio)} --> {_tiempo_srt(fin)}\n{texto}\n"
        for numero, (inicio, fin, texto) in enumerate(subtitulos, start=1)
    ]
    return "\n".join(bloques)


# ═══ Montos y cifras en español ═════════════════════════════════════════════════
# XTTS no reconoce «S/» y, en español, toma la coma como decimal: lee «1,500» como «uno punto cinco».

_UNIDADES = (
    "cero", "uno", "dos", "tres", "cuatro", "cinco", "seis", "siete", "ocho", "nueve", "diez",
    "once", "doce", "trece", "catorce", "quince", "dieciséis", "diecisiete", "dieciocho", "diecinueve",
    "veinte", "veintiuno", "veintidós", "veintitrés", "veinticuatro", "veinticinco", "veintiséis",
    "veintisiete", "veintiocho", "veintinueve",
)
_DECENAS = ("", "", "", "treinta", "cuarenta", "cincuenta", "sesenta", "setenta", "ochenta", "noventa")
_CENTENAS = (
    "", "ciento", "doscientos", "trescientos", "cuatrocientos", "quinientos",
    "seiscientos", "setecientos", "ochocientos", "novecientos",
)
_ESCALAS = (("", ""), ("millón", "millones"), ("billón", "billones"), ("trillón", "trillones"))


def numero_en_palabras(n: int, *, apocope: bool = False) -> str:
    """Entero ≥ 0 en palabras (escala larga). Con `apocope`, el «uno» final pasa a «un» («veintiún soles»)."""
    if n < 0 or n >= 1000 ** (2 * len(_ESCALAS)):
        raise ValueError(f"Número fuera de rango: {n}")
    if n == 0:
        return "cero"
    grupos: list[str] = []
    for nivel in range(len(_ESCALAS)):
        n, grupo = divmod(n, 1_000_000)
        if grupo:
            grupos.append(_grupo_de_millon(grupo, nivel, apocope and nivel == 0))
        if n == 0:
            break
    return " ".join(reversed(grupos))


def _grupo_de_millon(grupo: int, nivel: int, apocope: bool) -> str:
    """Un grupo de 1 a 999 999 seguido de su escala («millones», «billones»…)."""
    if nivel == 0:
        return _hasta_un_millon(grupo, apocope)
    singular, plural = _ESCALAS[nivel]
    return f"un {singular}" if grupo == 1 else f"{_hasta_un_millon(grupo, True)} {plural}"


def _hasta_un_millon(n: int, apocope: bool) -> str:
    miles, resto = divmod(n, 1000)
    partes: list[str] = []
    if miles == 1:
        partes.append("mil")
    elif miles:
        partes.append(f"{_apocopar(_hasta_mil(miles))} mil")
    if resto:
        palabras = _hasta_mil(resto)
        partes.append(_apocopar(palabras) if apocope else palabras)
    return " ".join(partes)


def _hasta_mil(n: int) -> str:
    """1 a 999."""
    if n == 100:
        return "cien"
    centenas, resto = divmod(n, 100)
    partes = [_CENTENAS[centenas]] if centenas else []
    if resto >= 30:
        decenas, unidades = divmod(resto, 10)
        partes.append(_DECENAS[decenas] + (f" y {_UNIDADES[unidades]}" if unidades else ""))
    elif resto:
        partes.append(_UNIDADES[resto])
    return " ".join(partes)


def _apocopar(palabras: str) -> str:
    """«uno» → «un» y «veintiuno» → «veintiún», como exige el español ante «mil», «millones» o un masculino."""
    if palabras.endswith("veintiuno"):
        return palabras[: -len("veintiuno")] + "veintiún"
    if palabras == "uno" or palabras.endswith(" uno"):
        return palabras[: -len("uno")] + "un"
    return palabras


@dataclass(frozen=True)
class Moneda:
    singular: str
    plural: str
    fraccion_singular: str
    fraccion_plural: str


SOLES = Moneda("sol", "soles", "céntimo", "céntimos")
DOLARES = Moneda("dólar", "dólares", "centavo", "centavos")
_CIFRA = r"\d(?:[\d.,]*\d)?"
_PATRON_SOLES = re.compile(rf"(?<![\w/])S/\.?\s*({_CIFRA})", re.IGNORECASE)
_PATRON_DOLARES = re.compile(rf"(?<!\w)(?:US\s?\$|USD\s?|\$)\s*({_CIFRA})", re.IGNORECASE)
_PATRON_MILES_CON_COMA = re.compile(r"(?<![\d.,])\d{1,3}(?:,\d{3})+(?:\.\d+)?(?![\d,])")


def parsear_monto(cifra: str) -> tuple[int, int] | None:
    """«1,500.50», «1.500,50», «1500» o «12.5» → (entero, céntimos); None si no se entiende.

    Si solo hay un tipo de separador, aparece una vez y no lo siguen exactamente 3 cifras, es decimal;
    en cualquier otro caso es separador de miles («1,500» y «1.500» valen mil quinientos).
    """
    punto, coma = cifra.rfind("."), cifra.rfind(",")
    decimal = ""
    if punto >= 0 and coma >= 0:
        decimal = "." if punto > coma else ","
    elif punto >= 0 or coma >= 0:
        separador = "." if punto >= 0 else ","
        if cifra.count(separador) == 1 and len(cifra.rsplit(separador, 1)[1]) != 3:
            decimal = separador
    entero, _, fraccion = cifra.rpartition(decimal) if decimal else (cifra, "", "")
    entero = entero.replace(".", "").replace(",", "")
    if not entero.isdecimal() or (fraccion and not fraccion.isdecimal()):
        return None
    return int(entero), int((fraccion + "00")[:2]) if fraccion else 0


def monto_en_palabras(entero: int, centimos: int, moneda: Moneda) -> str:
    """(1500, 50, SOLES) → «mil quinientos soles con cincuenta céntimos»; (1, 0, SOLES) → «un sol»."""
    partes: list[str] = []
    if entero or not centimos:
        de = " de" if entero and entero % 1_000_000 == 0 else ""  # «un millón de soles»
        nombre = moneda.singular if entero == 1 else moneda.plural
        partes.append(f"{numero_en_palabras(entero, apocope=True)}{de} {nombre}")
    if centimos:
        nombre = moneda.fraccion_singular if centimos == 1 else moneda.fraccion_plural
        partes.append(f"{numero_en_palabras(centimos, apocope=True)} {nombre}")
    return " con ".join(partes)


def _leer_monto(coincidencia: re.Match[str], moneda: Moneda) -> str:
    monto = parsear_monto(coincidencia.group(1))
    if monto is None:
        return coincidencia.group(0)
    try:
        return monto_en_palabras(*monto, moneda)
    except ValueError:
        return coincidencia.group(0)


def normalizar_cifras(texto: str) -> tuple[str, int]:
    """Montos en soles y dólares → palabras; «1,500» → «1500» para que XTTS lo lea como mil quinientos."""
    total = 0
    for patron, moneda in ((_PATRON_SOLES, SOLES), (_PATRON_DOLARES, DOLARES)):
        texto, cambios = patron.subn(lambda m, moneda=moneda: _leer_monto(m, moneda), texto)
        total += cambios
    texto, cambios = _PATRON_MILES_CON_COMA.subn(lambda m: m.group(0).replace(",", ""), texto)
    return texto, total + cambios


# ═══ Pronunciación ══════════════════════════════════════════════════════════════

LARGO_MAXIMO_REGLA = 100


@dataclass(frozen=True)
class Regla:
    """«texto = lectura»: cada aparición de `texto` (palabra completa, sin distinguir mayúsculas) se lee como `lectura`."""

    texto: str
    lectura: str


def leer_reglas(fuente: str) -> tuple[list[Regla], list[str]]:
    """Reglas separadas por «;» o por líneas; una línea que empieza con «#» es un comentario.

    Si un texto se repite, gana la última regla: así el formulario puede corregir lo que dice el archivo.
    """
    reglas: dict[str, Regla] = {}
    avisos: list[str] = []
    for linea in fuente.splitlines():
        if linea.lstrip().startswith("#"):
            continue
        for crudo in linea.split(";"):
            if not crudo.strip():
                continue
            texto, igual, lectura = (" ".join(parte.split()) for parte in crudo.partition("="))
            if not igual or not texto:
                avisos.append(f"Regla ignorada (formato «texto = lectura»): «{crudo.strip()}»")
                continue
            if len(texto) > LARGO_MAXIMO_REGLA:
                avisos.append(f"Regla ignorada (más de {LARGO_MAXIMO_REGLA} caracteres): «{texto[:40]}…»")
                continue
            texto = unicodedata.normalize("NFC", texto)
            reglas.pop(texto.lower(), None)
            reglas[texto.lower()] = Regla(texto, unicodedata.normalize("NFC", lectura))
    return list(reglas.values()), avisos


class _NodoTrie:
    __slots__ = ("hijos", "fin")

    def __init__(self) -> None:
        self.hijos: dict[str, _NodoTrie] = {}
        self.fin = False


def _patron_de_trie(nodo: _NodoTrie) -> str:
    """Regex equivalente a la trie: prefijos comunes factorizados; el «?» codicioso prueba primero la clave más larga."""
    ramas = [re.escape(caracter) + _patron_de_trie(hijo) for caracter, hijo in sorted(nodo.hijos.items())]
    if not ramas:
        return ""
    cuerpo = ramas[0] if len(ramas) == 1 else "(?:" + "|".join(ramas) + ")"
    return f"(?:{cuerpo})?" if nodo.fin else cuerpo


class Pronunciador:
    """Aplica todas las reglas en una sola pasada; si dos empiezan igual («Machu» / «Machu Picchu»), gana la más larga.

    Los textos se guardan en una trie que se compila a una única regex con los prefijos comunes factorizados:
    el trabajo por posición depende del largo de las claves, no de cuántas reglas haya.
    """

    def __init__(self, reglas: Sequence[Regla]) -> None:
        self.cantidad = len(reglas)
        self._lecturas = {regla.texto.lower(): regla.lectura for regla in reglas}
        raiz = _NodoTrie()
        for clave in self._lecturas:
            nodo = raiz
            for caracter in clave:
                nodo = nodo.hijos.setdefault(caracter, _NodoTrie())
            nodo.fin = True
        self._patron = (
            re.compile(rf"(?<!\w)(?:{_patron_de_trie(raiz)})(?!\w)", re.IGNORECASE) if self._lecturas else None
        )

    def aplicar(self, texto: str) -> tuple[str, int]:
        """Texto con las lecturas aplicadas y cuántos reemplazos hubo."""
        texto = re.sub(r"[ \t]+", " ", unicodedata.normalize("NFC", texto))
        if self._patron is None:
            return texto, 0
        return self._patron.subn(lambda m: self._lecturas.get(m.group(0).lower(), m.group(0)), texto)


# ═══ Comparar lo que se pidió con lo que se oyó (control de calidad con Whisper) ═


def distancia_edicion(a: str, b: str) -> int:
    """Distancia de Levenshtein con el algoritmo de vectores de bits de Myers (versión de Hyyrö): cada columna
    de la matriz de programación dinámica cabe en un entero de Python, así el costo es O(len(b)) operaciones
    sobre enteros de len(a) bits en lugar de O(len(a)·len(b)) pasos en Python."""
    if len(a) < len(b):
        a, b = b, a
    m = len(b)
    if m == 0:
        return len(a)
    mascara, alto = (1 << m) - 1, 1 << (m - 1)
    posiciones: dict[str, int] = {}
    for i, caracter in enumerate(b):
        posiciones[caracter] = posiciones.get(caracter, 0) | (1 << i)
    positivos, negativos, distancia = mascara, 0, m
    for caracter in a:
        iguales = posiciones.get(caracter, 0)
        x_vertical = iguales | negativos
        x_horizontal = (((iguales & positivos) + positivos) ^ positivos) | iguales
        horizontal_positivo = negativos | (~(x_horizontal | positivos) & mascara)
        horizontal_negativo = positivos & x_horizontal
        if horizontal_positivo & alto:
            distancia += 1
        elif horizontal_negativo & alto:
            distancia -= 1
        horizontal_positivo = ((horizontal_positivo << 1) | 1) & mascara  # fila 0: D[0][j] = j
        horizontal_negativo = (horizontal_negativo << 1) & mascara
        positivos = horizontal_negativo | (~(x_vertical | horizontal_positivo) & mascara)
        negativos = horizontal_positivo & x_vertical
    return distancia


def normalizar_para_comparar(texto: str, idioma: str = "") -> str:
    """Minúsculas, sin tildes ni signos y con espacios simples; en español, las cifras pasan a palabras
    (Whisper suele escribir «1500» donde se dijo «mil quinientos»)."""
    if idioma.startswith("es"):
        texto = re.sub(r"\d+", lambda m: numero_en_palabras(int(m.group(0))) if len(m.group(0)) <= 15 else m.group(0), texto)
    sin_tildes = "".join(c for c in unicodedata.normalize("NFKD", texto.casefold()) if not unicodedata.combining(c))
    return " ".join(re.sub(r"[^\w\s]|_", " ", sin_tildes).split())


def tasa_error_caracteres(referencia: str, transcripcion: str, idioma: str = "") -> float:
    """CER: ediciones de caracteres para pasar de lo pedido a lo que se oyó, sobre el largo de lo pedido."""
    pedido = normalizar_para_comparar(referencia, idioma)
    oido = normalizar_para_comparar(transcripcion, idioma)
    return distancia_edicion(pedido, oido) / max(1, len(pedido))


# ─── Ejecución ───
print("✅ Funciones de texto cargadas.")

In [ ]:
# @title 3 · Preparar tus grabaciones para el entrenamiento { display-mode: "form" }
# @markdown **Uso asumido por defecto: no comercial.** Es lo único que permite la licencia de XTTS-v2 ([CPML](https://tts-hub.github.io/cpml)) para el modelo original, el que afines y sus audios; con la casilla marcada, el cuaderno la acepta en tu nombre.
uso_no_comercial = True  # @param {type:"boolean"}
# @markdown Escribe la **carpeta de tu Drive** con tus grabaciones (o un solo audio largo). Whisper las transcribe y las corta en frases de hasta 11 segundos, el formato con que aprende el modelo.
audios_de_entrenamiento = "Audios/Mi voz"  # @param {type:"string"}
idioma_de_los_audios = "Español"  # @param ["Español", "English", "Português", "Français", "Italiano", "Deutsch", "Nederlands", "Polski", "Türkçe", "Русский", "Čeština", "Magyar", "العربية", "हिन्दी"]
quitar_ruido_de_fondo = False  # @param {type:"boolean"}
# @markdown *Actívalo solo si se oye un ruido constante (ventilador, aire acondicionado). Se incluyen las subcarpetas; la primera vez se descarga Whisper large-v3 (~3 GB).*

"""Celda 3 (entrenamiento) · Datos: grabaciones → WAV de 22 050 Hz → Whisper (marcas de tiempo por palabra) →
frases de 0,6 a 11 s con su texto → metadata_train.csv y metadata_eval.csv en el formato «coqui» de coqui-tts."""

import random
from collections.abc import Sequence
from dataclasses import dataclass
from pathlib import Path

if "leer_wav" not in globals() or "dividir_texto" not in globals():
    raise RuntimeError("Ejecuta primero las celdas 1, 2a, 2b y 2c.")
if globals().get("MOTOR_ELEGIDO") != "xtts":
    raise ErrorUsuario("El entrenamiento es solo para XTTS-v2: elígelo en la celda 1 (en un entorno nuevo) y vuelve a empezar.")
confirmar_licencia(uso_no_comercial)

CARPETA_ENTRENAMIENTO = CARPETA_TRABAJO / "entrenamiento"
FRECUENCIA_ENTRENAMIENTO = 22050  # la que espera el entrenador de XTTS-v2
SEGUNDOS_MAXIMOS_CLIP = 11.0  # el entrenador descarta audios de más de ~11,6 s
SEGUNDOS_MINIMOS_CLIP = 0.6
CARACTERES_MAXIMOS_CLIP = 200  # y textos de más de 200 caracteres
MARGEN_CLIP = 0.2  # segundos de aire antes y después de cada frase, sin invadir la palabra vecina
FINALES_DE_FRASE = (".", "!", "?", "…", "。", "！", "？")
CORTES_SUAVES = (",", ";", ":", "—", "–")
SONORIDAD_ENTRENAMIENTO = -20.0  # LUFS: todas las frases con el mismo volumen
PROPORCION_EVALUACION = 0.15
MINIMO_DE_FRASES = 10
MINUTOS_RECOMENDADOS = 10
MAXIMO_ARCHIVOS_ENTRENAMIENTO = 300
SEGUNDOS_MAXIMOS_POR_ARCHIVO = 3 * 3600
MODELO_WHISPER_ENTRENAMIENTO = "large-v3"  # el más preciso: los errores de transcripción se aprenden
HABLANTE = "mi_voz"


@dataclass(frozen=True)
class Palabra:
    inicio: float
    fin: float
    texto: str


@dataclass(frozen=True)
class Clip:
    inicio: float
    fin: float
    texto: str


@dataclass(frozen=True)
class DatosEntrenamiento:
    carpeta: Path
    csv_entrenamiento: Path
    csv_evaluacion: Path
    idioma: str
    frases: int
    segundos: float


def _mejor_corte(palabras: Sequence[Palabra], grupo: Sequence[int]) -> int:
    """Posición del grupo tras la cual cortar una frase demasiado larga: la pausa más larga entre palabras, con
    preferencia por comas y similares, sin dejar un primer trozo de menos de 1 s."""
    inicio = palabras[grupo[0]].inicio
    mejor, puntaje_mejor = len(grupo) - 1, -1.0
    for posicion in range(len(grupo) - 1):
        actual, siguiente = palabras[grupo[posicion]], palabras[grupo[posicion + 1]]
        if actual.fin - inicio < 1.0:
            continue
        puntaje = (siguiente.inicio - actual.fin) + (0.25 if actual.texto.rstrip().endswith(CORTES_SUAVES) else 0.0)
        if puntaje > puntaje_mejor:
            mejor, puntaje_mejor = posicion, puntaje
    return mejor


def _clip(palabras: Sequence[Palabra], grupo: Sequence[int], duracion: float) -> Clip | None:
    primera, ultima = palabras[grupo[0]], palabras[grupo[-1]]
    fin_anterior = palabras[grupo[0] - 1].fin if grupo[0] > 0 else 0.0
    inicio_siguiente = palabras[grupo[-1] + 1].inicio if grupo[-1] + 1 < len(palabras) else duracion
    inicio = max(primera.inicio - MARGEN_CLIP, (fin_anterior + primera.inicio) / 2, 0.0)
    fin = min(ultima.fin + MARGEN_CLIP, (ultima.fin + inicio_siguiente) / 2, duracion)
    texto = " ".join("".join(palabras[i].texto for i in grupo).split())
    if fin - inicio < SEGUNDOS_MINIMOS_CLIP or len(texto) > CARACTERES_MAXIMOS_CLIP or not any(c.isalnum() for c in texto):
        return None
    return Clip(inicio, fin, texto)


def cortar_en_frases(palabras: Sequence[Palabra], duracion: float) -> list[Clip]:
    """Agrupa las palabras de Whisper en frases: cierra en . ! ? y, si una frase se pasaría de 11 s, la corta antes
    en su mejor pausa interna (el formateador de coqui-tts las deja largas y el entrenador las descarta)."""
    limite = SEGUNDOS_MAXIMOS_CLIP - 2 * MARGEN_CLIP
    grupos: list[list[int]] = []
    actual: list[int] = []
    for indice, palabra in enumerate(palabras):
        while actual and palabra.fin - palabras[actual[0]].inicio > limite:
            corte = _mejor_corte(palabras, actual)
            grupos.append(actual[: corte + 1])
            actual = actual[corte + 1 :]
        actual.append(indice)
        if palabra.texto.rstrip().endswith(FINALES_DE_FRASE):
            grupos.append(actual)
            actual = []
    if actual:
        grupos.append(actual)
    return [clip for grupo in grupos if (clip := _clip(palabras, grupo, duracion)) is not None]


def filtros_entrenamiento(quitar_ruido: bool) -> str:
    """Sin recortar silencios ni comprimir el volumen: las frases se cortan después, con las marcas de Whisper."""
    return ",".join(["highpass=f=70"] + (["afftdn=nr=10:nf=-30"] if quitar_ruido else []))


def texto_para_csv(texto: str) -> str:
    """El formato «coqui» separa columnas con «|» y lee las comillas como delimitadores: no pueden ir en el texto."""
    return " ".join(texto.replace("|", ",").replace('"', "").split())


def escribir_metadatos(ruta: Path, filas: Sequence[tuple[str, str]]) -> None:
    lineas = ["audio_file|text|speaker_name"] + [f"{audio}|{texto_para_csv(texto)}|{HABLANTE}" for audio, texto in filas]
    ruta.write_text("\n".join(lineas) + "\n", encoding="utf-8")


def transcribir_palabras(wav: Path, idioma: str) -> list[Palabra]:
    trozos, _ = obtener_whisper(MODELO_WHISPER_ENTRENAMIENTO).transcribe(
        str(wav), language=idioma.split("-")[0], word_timestamps=True, vad_filter=True, beam_size=5
    )
    return [Palabra(p.start, p.end, p.word) for trozo in trozos for p in (trozo.words or [])]


def preparar_datos(entrada: str, idioma: str, quitar_ruido: bool) -> DatosEntrenamiento:
    from TTS.tts.layers.xtts.tokenizer import multilingual_cleaners

    montar_drive()
    origen = resolver_en_drive(entrada, extensiones=EXTENSIONES_MEDIOS, incluir_carpetas=True)
    archivos = listar_medios(origen, MAXIMO_ARCHIVOS_ENTRENAMIENTO, subcarpetas=True)
    carpeta = preparar_carpeta(CARPETA_ENTRENAMIENTO / "datos", vaciar=True)
    convertidos = preparar_carpeta(CARPETA_ENTRENAMIENTO / "convertidos", vaciar=True)
    preparar_carpeta(carpeta / "wavs")
    print(f"🎙️ {len(archivos)} archivo(s): transcribiendo con Whisper {MODELO_WHISPER_ENTRENAMIENTO}…")
    filas: list[tuple[str, str]] = []
    segundos = 0.0
    try:
        for indice, archivo in enumerate(archivos, start=1):
            wav = convertidos / f"{indice:03d}.wav"
            preparar_referencia(
                archivo, wav, filtros=filtros_entrenamiento(quitar_ruido), max_segundos=SEGUNDOS_MAXIMOS_POR_ARCHIVO,
                frecuencia=FRECUENCIA_ENTRENAMIENTO,
            )
            muestras, frecuencia = leer_wav(wav)
            clips = cortar_en_frases(transcribir_palabras(wav, idioma), muestras.size / frecuencia)
            base = f"{indice:03d}_{nombre_seguro(archivo.stem, por_defecto='audio')[:40]}"
            for numero, clip in enumerate(clips):
                audio = muestras[int(clip.inicio * frecuencia) : int(clip.fin * frecuencia)]
                audio = aplicar_fundidos(nivelar_volumen(audio, frecuencia, SONORIDAD_ENTRENAMIENTO), frecuencia)
                relativo = f"wavs/{base}_{numero:05d}.wav"
                escribir_wav(carpeta / relativo, audio, frecuencia)
                filas.append((relativo, multilingual_cleaners(clip.texto, idioma)))
                segundos += audio.size / frecuencia
            print(f"   🎧 {archivo.name}: {len(clips)} frase(s)")
    finally:
        liberar_whisper()  # la GPU queda libre para entrenar
    if len(filas) < MINIMO_DE_FRASES:
        raise ErrorUsuario(
            f"Solo salieron {len(filas)} frase(s) útiles; hacen falta al menos {MINIMO_DE_FRASES} "
            "(lo ideal son 10–30 minutos de tu voz)."
        )
    orden = list(range(len(filas)))
    random.Random(0).shuffle(orden)  # reparto repetible
    cantidad_evaluacion = max(1, round(len(filas) * PROPORCION_EVALUACION))
    evaluacion, entrenamiento = sorted(orden[:cantidad_evaluacion]), sorted(orden[cantidad_evaluacion:])
    csv_entrenamiento, csv_evaluacion = carpeta / "metadata_train.csv", carpeta / "metadata_eval.csv"
    escribir_metadatos(csv_entrenamiento, [filas[i] for i in entrenamiento])
    escribir_metadatos(csv_evaluacion, [filas[i] for i in evaluacion])
    print(f"✅ {len(filas)} frases · {formatear_duracion(segundos)} de voz ({len(entrenamiento)} para entrenar, {len(evaluacion)} para evaluar).")
    if segundos < MINUTOS_RECOMENDADOS * 60:
        print(f"⚠️ Con menos de {MINUTOS_RECOMENDADOS} minutos el afinado mejora poco: si puedes, agrega más grabaciones.")
    print("   Revisa algunas transcripciones (los errores se aprenden):")
    for audio, texto in filas[:3]:
        print(f"   • {audio}: {texto}")
    return DatosEntrenamiento(carpeta, csv_entrenamiento, csv_evaluacion, idioma, len(filas), segundos)


# ─── Ejecución ───
DATOS_ENTRENAMIENTO = None
DATOS_ENTRENAMIENTO = preparar_datos(audios_de_entrenamiento, IDIOMAS[idioma_de_los_audios], quitar_ruido_de_fondo)

In [ ]:
# @title 4 · Entrenar XTTS-v2 con tu voz { display-mode: "form" }
# @markdown Afina el modelo con las frases de la celda 3. En una T4 tarda de 10 a 40 minutos según la cantidad de audio; la primera vez descarga además el XTTS-v2 original (~2 GB). No detengas la celda: el entrenador cerraría la sesión.
epocas = 6  # @param {type:"slider", min:1, max:20, step:1}
# @markdown *Pasadas completas por tus frases. Con pocos minutos de audio, más épocas (10–15); con mucho, menos (4–6). Demasiadas pueden empeorar el resultado.*
tamano_de_lote = 2  # @param {type:"slider", min:1, max:4, step:1}
# @markdown *Frases por paso. Si aparece «CUDA out of memory», bájalo a 1.*
acumulacion_de_gradiente = 1  # @param {type:"slider", min:1, max:16, step:1}
# @markdown *Simula un lote más grande sin usar más memoria (lote efectivo = lote × acumulación).*

"""Celda 4 (entrenamiento) · Entrenar: el afinado oficial de coqui-tts (train_gpt de su demo) con tus frases y los
archivos originales de XTTS-v2 tomados de la caché de Hugging Face, como los descarga coqui-tts."""

import shutil
import time
from dataclasses import dataclass
from pathlib import Path

if globals().get("DATOS_ENTRENAMIENTO") is None:
    raise RuntimeError("Ejecuta primero la celda 3.")

ORIGINALES = ("model.pth", "config.json", "vocab.json", "mel_stats.pth", "dvae.pth", "speakers_xtts.pth")
CARPETA_ORIGINALES = "XTTS_v2.0_original_model_files"  # donde train_gpt busca los archivos antes de descargarlos


@dataclass(frozen=True)
class ModeloEntrenado:
    checkpoint: Path
    configuracion: Path
    vocabulario: Path
    originales: Path
    datos: DatosEntrenamiento
    epocas: int


def repositorio_xtts() -> str:
    """Repositorio de Hugging Face del que coqui-tts descarga XTTS-v2 (en la versión fijada, tts-hub/XTTS-v2)."""
    from TTS.utils.manage import ModelManager

    return ModelManager().models_dict["tts_models"]["multilingual"]["multi-dataset"]["xtts_v2"]["repo_id"]


def archivo_original(nombre: str) -> Path:
    """Un archivo del XTTS-v2 original desde la caché de Hugging Face: lo ya descargado no se vuelve a bajar."""
    from huggingface_hub import hf_hub_download

    repositorio = repositorio_xtts()
    try:
        return Path(hf_hub_download(repositorio, nombre))
    except Exception as error:  # noqa: BLE001 — sin red o sin acceso: mejor un mensaje claro que un archivo roto
        raise ErrorUsuario(
            f"No se pudo descargar «{nombre}» de {repositorio} ({type(error).__name__}: {error}). Revisa la conexión "
            "y vuelve a ejecutar esta celda."
        ) from None


def preparar_archivos_originales(salida: Path) -> Path:
    """Si train_gpt no encuentra los archivos originales en su carpeta, los descarga de coqui/XTTS-v2 sin comprobar
    errores HTTP (una página de error quedaría guardada como model.pth). Se le dejan listos: el modelo como enlace a la
    caché y el resto copiado, porque el entrenador puede reescribirlos."""
    destino = preparar_carpeta(salida / "run" / "training" / CARPETA_ORIGINALES)
    print("📥 Archivos originales de XTTS-v2 (la primera vez se descargan, ~2 GB)…")
    for nombre in ORIGINALES:
        origen = archivo_original(nombre)
        if nombre == "model.pth":
            (destino / nombre).symlink_to(origen)
        else:
            shutil.copyfile(origen, destino / nombre)
    return destino


def buscar_checkpoint(carpeta: Path) -> Path:
    mejor = carpeta / "best_model.pth"
    if mejor.is_file():
        return mejor
    candidatos = sorted(carpeta.glob("*.pth"), key=lambda ruta: ruta.stat().st_mtime)
    if not candidatos:
        raise ErrorUsuario("El entrenamiento terminó sin guardar ningún modelo: revisa los mensajes de arriba.")
    return candidatos[-1]


def entrenar(datos: DatosEntrenamiento, epocas: int, lote: int, acumulacion: int) -> ModeloEntrenado:
    liberar_whisper()  # toda la GPU para el entrenamiento
    salida = preparar_carpeta(CARPETA_ENTRENAMIENTO / "salida", vaciar=True)
    originales = preparar_archivos_originales(salida)
    from TTS.demos.xtts_ft_demo.utils.gpt_train import train_gpt

    print(
        f"🏋️ Entrenando con {datos.frases} frases ({formatear_duracion(datos.segundos)}) · {epocas} época(s) · lote "
        f"{lote} × {acumulacion}. Los mensajes que siguen son del entrenador de coqui-tts."
    )
    inicio = time.perf_counter()
    try:
        configuracion, _, vocabulario, carpeta, _ = train_gpt(
            datos.idioma,
            int(epocas),
            int(lote),
            int(acumulacion),
            str(datos.csv_entrenamiento),
            str(datos.csv_evaluacion),
            output_path=str(salida),
        )
    except SystemExit:  # el entrenador de coqui-tts termina con sys.exit(1) ante cualquier error
        raise ErrorUsuario(
            "El entrenamiento se interrumpió (el detalle está arriba). Si dice «CUDA out of memory», baja "
            "«tamano_de_lote» a 1 y vuelve a ejecutar esta celda."
        ) from None
    checkpoint = buscar_checkpoint(Path(carpeta))
    print(f"✅ Entrenamiento terminado en {formatear_duracion(time.perf_counter() - inicio)}. Guárdalo con la celda 5.")
    return ModeloEntrenado(checkpoint, Path(configuracion), Path(vocabulario), originales, datos, int(epocas))


# ─── Ejecución ───
MODELO_ENTRENADO = None
MODELO_ENTRENADO = entrenar(DATOS_ENTRENAMIENTO, epocas, tamano_de_lote, acumulacion_de_gradiente)

In [ ]:
# @title 5 · Guardar tu modelo afinado en Drive { display-mode: "form" }
nombre_del_modelo = "mi_voz_afinada"  # @param {type:"string"}
# @markdown *Se guarda en `Mi unidad/Clonador de voz/modelos/<nombre>` (~1 GB, en media precisión). Para usarlo, en el Clonador de voz: celda 3 → escribe este nombre en `modelo_afinado` y ejecútala; luego crea tu voz en la celda 4 (sirve el `referencia.wav` que se guarda junto al modelo).*

"""Celda 5 (entrenamiento) · Guardar: compacta el checkpoint del entrenador (solo los pesos de inferencia, en float16) y lo copia a
Drive con su configuración, su vocabulario, las voces incluidas y un audio de referencia hecho con tus frases."""

import json
import shutil
import time
from pathlib import Path

import numpy as np

if globals().get("MODELO_ENTRENADO") is None:
    raise RuntimeError("Ejecuta primero la celda 4.")

CLAVES_DEL_ENTRENADOR = ("dvae.", "torch_mel_spectrogram_style_encoder.", "torch_mel_spectrogram_dvae.")
SEGUNDOS_DE_REFERENCIA = 30.0  # la celda 4 da su mejor parecido con 30–60 s


def compactar_checkpoint(origen: Path, destino: Path) -> int:
    """Del checkpoint del entrenador (pesos, optimizador, partes que solo sirven para entrenar) deja los pesos del
    modelo de inferencia, en media precisión: el mismo resultado que Xtts.get_compatible_checkpoint_state_dict,
    pero en ~1 GB en lugar de varios."""
    import torch

    checkpoint = torch.load(origen, map_location="cpu", weights_only=False)  # archivo propio, recién entrenado
    estado = {}
    for clave, valor in checkpoint["model"].items():
        clave = clave.removeprefix("xtts.")
        if clave.startswith(CLAVES_DEL_ENTRENADOR):
            continue
        estado[clave] = valor.half() if valor.is_floating_point() else valor
    del checkpoint
    torch.save({"model": estado}, destino)
    return len(estado)


def construir_referencia(datos: DatosEntrenamiento, destino: Path) -> float:
    """Une las frases con más palabras del entrenamiento hasta ~30 s: una referencia para crear tu voz con el modelo
    afinado en la celda 4 del clonador (una sola frase sería demasiado corta)."""
    lineas = datos.csv_entrenamiento.read_text(encoding="utf-8").splitlines()[1:]
    filas = sorted((linea.split("|")[:2] for linea in lineas if linea), key=lambda fila: len(fila[1].split()), reverse=True)
    partes, total = [], 0.0
    for audio, _ in filas:
        muestras, frecuencia = leer_wav(datos.carpeta / audio)
        partes += [muestras, np.zeros(round(0.3 * frecuencia), np.float32)]
        total += muestras.size / frecuencia
        if total >= SEGUNDOS_DE_REFERENCIA:
            break
    escribir_wav(destino, np.concatenate(partes), FRECUENCIA_ENTRENAMIENTO)
    return total


def carpeta_libre(padre: Path, base: str) -> Path:
    candidata, numero = padre / base, 2
    while candidata.exists():
        candidata, numero = padre / f"{base}_{numero}", numero + 1
    return candidata


def guardar_modelo_afinado(modelo: ModeloEntrenado, nombre: str) -> Path:
    local = preparar_carpeta(CARPETA_ENTRENAMIENTO / "modelo_final", vaciar=True)
    print("📦 Compactando el modelo…")
    claves = compactar_checkpoint(modelo.checkpoint, local / "model.pth")
    shutil.copyfile(modelo.configuracion, local / "config.json")
    shutil.copyfile(modelo.vocabulario, local / "vocab.json")
    shutil.copyfile(modelo.originales / "speakers_xtts.pth", local / "speakers_xtts.pth")  # voces incluidas (diálogos)
    construir_referencia(modelo.datos, local / "referencia.wav")
    informacion = {
        "idioma": modelo.datos.idioma,
        "frases": modelo.datos.frases,
        "minutos_de_voz": round(modelo.datos.segundos / 60, 1),
        "epocas": modelo.epocas,
        "parametros": claves,
        "coqui_tts": VERSION_COQUI_TTS,
        "creado": time.strftime("%Y-%m-%d %H:%M"),
    }
    (local / "info.json").write_text(json.dumps(informacion, ensure_ascii=False, indent=1), encoding="utf-8")
    destino = carpeta_libre(carpeta_del_proyecto("modelos"), nombre_seguro(nombre, por_defecto="mi_voz_afinada"))
    tamano = sum(archivo.stat().st_size for archivo in local.iterdir()) / 1e9
    print(f"💾 Copiando {tamano:.1f} GB a Drive…")
    shutil.copytree(local, destino)
    print(f"✅ Modelo guardado en {describir_ruta(destino)}.")
    print(
        f"   Para usarlo en el Clonador de voz: celda 3 → modelo_afinado = «{destino.name}» → ejecútala; en la celda 4 puedes usar "
        f"«{describir_ruta(destino / 'referencia.wav').removeprefix('Mi unidad/')}» como audio."
    )
    return destino


# ─── Ejecución ───
MODELO_AFINADO_GUARDADO = guardar_modelo_afinado(MODELO_ENTRENADO, nombre_del_modelo)

### Si algo no sale bien

| Problema | Qué hacer |
|---|---|
| «Solo salieron N frases útiles» | Hacen falta más grabaciones (lo ideal son 10–30 minutos). Revisa que la carpeta tenga audio con tu voz y no solo silencio o música. |
| Las transcripciones tienen errores | El modelo aprende lo que dice el texto: graba con buena dicción, sin ruido, y activa `quitar_ruido_de_fondo` si hace falta. |
| «CUDA out of memory» | En la celda 4 baja `tamano_de_lote` a 1 (y, si quieres, sube `acumulacion_de_gradiente`). |
| Detuviste la celda 4 a mitad del entrenamiento | El entrenador de coqui-tts cierra la sesión de Colab al interrumpirlo: empieza otra vez desde la celda 1. |
| El resultado suena peor que sin afinar | Prueba con menos `epocas` (4–6) o con grabaciones más limpias y variadas. |
| No ves el modelo en el clonador | Escribe en `modelo_afinado` el nombre exacto que imprimió la celda 5 (si ya existía uno con ese nombre, se guarda como `nombre_2`). |